# Podnájom alebo hypotéka? – jeden balík
Notebook obsahuje všetko, čo projekt potrebuje – **dáta, kód modelu, váhy neurónovej siete** – a nič sa v ňom nesťahuje ani neinštaluje (stačí Google Colab: *Runtime → Run all*).

Čo robí:
1. **Majetok v oboch scenároch** pre zvolený počiatočný rok (časť A),
2. **rada modelu** pre zadanú situáciu (časť B),
3. **vyhodnotenie na histórii**: pre každý počiatočný rok odpoveď modelu len z dát do tohto roku vs. skutočný víťaz (časť C).

Dáta: Španielsko 1870–2025, ročne (JST Macrohistory do 2020, 2021–2025 predĺženie z ECB/BIS). Zdroje a slabiny: https://github.com/adibuj2804/rent-vs-buy/blob/main/data/SOURCES.md · správa: https://adibuj2804.github.io/rent-vs-buy/sprava.html

*Vzdelávací model, nie finančné poradenstvo.*

## 0 · Dáta, model, sieť (spusti bez zmien)

In [ ]:
# DÁTA: ročné rady pre Španielsko (vložené; stĺpec `source` hovorí, čo je JST a čo predĺženie)
import io, json
import numpy as np
import pandas as pd
DATA_CSV = '''year,iso,cpi,ltrate,housing_capgain,housing_rent_yd,eq_tr,source
1870,ESP,0.2823471260174294,11.194029850746269,,,,JST
1871,ESP,0.2830402048773203,9.917355371900827,,,,JST
1872,ESP,0.2914561624617089,11.560693641618498,,,,JST
1873,ESP,0.3141627460619417,22.607385079125848,,,,JST
1874,ESP,0.3103673142101587,23.346303501945524,,,,JST
1875,ESP,0.3036675518978807,17.59530791788856,,,,JST
1876,ESP,0.2995420824937686,23.715415019762844,,,,JST
1877,ESP,0.3024134091990306,22.988505747126435,,,,JST
1878,ESP,0.300994247724016,19.672131147540984,,,,JST
1879,ESP,0.3031064880589214,19.2,,,,JST
1880,ESP,0.3074299799944309,15.78,,,,JST
1881,ESP,0.3094795219468866,11.95,,,,JST
1882,ESP,0.3207519551332187,10.75,,,,JST
1883,ESP,0.3057220408735227,6.64,,,,JST
1884,ESP,0.2783949299706558,6.56,,,,JST
1885,ESP,0.2749790409355164,6.9,,,,JST
1886,ESP,0.2729295100726172,6.45,,,,JST
1887,ESP,0.2630234350816002,5.93,,,,JST
1888,ESP,0.2678056709192151,5.49,,,,JST
1889,ESP,0.2722463249680829,5.28,,,,JST
1890,ESP,0.2862514730788334,5.14,,,,JST
1891,ESP,0.2664393238926142,5.47,,,,JST
1892,ESP,0.2739542750224791,6.24,,,,JST
1893,ESP,0.2667809049911225,6.14,,,,JST
1894,ESP,0.2568748309073702,5.77,,,,JST
1895,ESP,0.2715631518926537,5.72,,,,JST
1896,ESP,0.2589243618819866,5.96,,,,JST
1897,ESP,0.2814692259635297,6.17,,,,JST
1898,ESP,0.3091379300852369,9.09,,,,JST
1899,ESP,0.31460334999125,6.39,,,,JST
1900,ESP,0.3303164369909255,5.29,,0.0374534018337726,0.0419478863477706,JST
1901,ESP,0.3309996208744967,5.53,0.0219212938100099,0.0367673709988594,0.0222680754959583,JST
1902,ESP,0.3234846590365821,4.78,-0.0387722477316856,0.0383725464344024,0.0438395552337169,JST
1903,ESP,0.333732326158566,4.45,0.0852419286966323,0.0354710482060909,0.0540446043014526,JST
1904,ESP,0.3398809298608811,4.59,-0.0958611592650413,0.0393602065742015,-0.0013100476935505,JST
1905,ESP,0.3415888691684836,4.34,-0.0798824056982994,0.0442525930702686,-0.057440310716629,JST
1906,ESP,0.3323659679713459,4.17,0.0072436057962477,0.0453192330896854,0.1047434359788894,JST
1907,ESP,0.3463711160492775,4.27,-0.0545091480016708,0.0481095090508461,0.0770718455314636,JST
1908,ESP,0.3368066220966839,4.16,0.0016653387574478,0.0488315671682357,0.0424919165670871,JST
1909,ESP,0.3323659679713459,4.09,0.0397191792726516,0.0459006316959857,0.0484555773437023,JST
1910,ESP,0.3354402761073622,4.18,-0.0074202502146363,0.0445692092180252,0.0242380499839782,JST
1911,ESP,0.3234846590365821,4.18,-0.0513659492135047,0.0484055653214454,0.021169040352106,JST
1912,ESP,0.3395393381012502,4.28,0.118662841618061,0.0393211208283901,0.0533804707229137,JST
1913,ESP,0.3415888691684836,4.4,0.0138699728995561,0.0394002757966518,0.0275302864611148,JST
1914,ESP,0.3405641029084553,4.22,-0.0099310958757996,0.0404210723936557,-0.0723688229918479,JST
1915,ESP,0.3730150420274868,4.47,-0.0252443980425596,0.0425890125334262,0.1292514503002166,JST
1916,ESP,0.4399664631888785,4.31,0.0636858791112899,0.039839282631874,0.0904203131794929,JST
1917,ESP,0.5127248958770176,4.31,0.1191159635782241,0.0431413352489471,0.1515799462795257,JST
1918,ESP,0.7450053309429933,4.12,0.3157175779342651,0.0339037850499153,0.1326514482498169,JST
1919,ESP,0.760718416753869,4.16,0.1021511182188987,0.0383278578519821,0.0214683096855878,JST
1920,ESP,0.7774562699094358,4.44,0.2349123954772949,0.0414667055010795,0.0640176609158515,JST
1921,ESP,0.6264739842794006,4.72,-0.3051118552684784,0.063663862645626,-0.0500439964234828,JST
1922,ESP,0.5479085497381593,4.62,-0.0461268201470375,0.066210076212883,0.2126449048519134,JST
1923,ESP,0.5520076130736815,4.53,0.0288283284753561,0.0650896057486534,0.1751825362443924,JST
1924,ESP,0.612127255047958,4.57,0.0486754104495048,0.0659363567829132,0.09121785312891,JST
1925,ESP,0.6216917492917946,4.58,0.1197633296251297,0.058090005069971,0.0510161370038986,JST
1926,ESP,0.5694286490215961,4.71,-0.0069159758277237,0.0597141534090042,0.1769116967916488,JST
1927,ESP,0.5588393898587377,4.61,-0.1009538918733596,0.0694886073470115,0.3053540885448456,JST
1928,ESP,0.5728445382693444,4.29,-0.1211601644754409,0.0816129818558693,0.2496404200792312,JST
1929,ESP,0.5888992167821371,4.34,0.0003528954403009,0.0903511121869087,0.0255135409533977,JST
1930,ESP,0.5909487492697648,4.48,0.4561940729618072,0.0595509856939315,0.0041283345781266,JST
1931,ESP,0.5947062177813288,5.53,-0.039851751178503,0.0613410286605358,-0.2420930117368698,JST
1932,ESP,0.5902655639269418,5.03,-0.1376501619815826,0.0716147497296333,-0.094573862850666,JST
1933,ESP,0.5639632297996959,4.83,-0.0088506443426013,0.0717686340212822,0.1209597215056419,JST
1934,ESP,0.5755772429347892,4.58,0.2996726632118225,0.0557883344590663,-0.0179753880947828,JST
1935,ESP,0.5885576247874126,4.19,-0.1116643846035003,0.0605347417294979,0.5931804776191711,JST
1936,ESP,0.5994884648404608,4.30755754627735,-0.0277693625539541,0.0751646161079406,-0.0659040510654449,JST
1937,ESP,0.6732716630360474,,0.2895363867282867,0.0583177357912063,0.0878115072846412,JST
1938,ESP,0.7409062573414761,,0.4061685800552368,0.0414938665926456,0.0878115072846412,JST
1939,ESP,0.8389422699892648,,-0.1154731214046478,0.0469346307218074,0.0878115072846412,JST
1940,ESP,0.9977810884319988,4.488330341113106,0.1429277062416076,0.0412889495491981,0.0978361517190933,JST
1941,ESP,1.19556104149745,4.426247648555936,0.1436333358287811,0.0362945534288883,0.3014000058174133,JST
1942,ESP,1.364305944831929,4.4145237832468815,0.0638992115855217,0.0343491435050964,0.1356231719255447,JST
1943,ESP,1.5620859101483209,4.41988950276243,0.1519818007946014,0.0299658477306365,-0.0723323598504066,JST
1944,ESP,1.6819836030175517,4.358722894192002,-0.1226040795445442,0.0343188792467117,0.0197665933519601,JST
1945,ESP,1.8298915796402755,4.349244318799609,0.2640551924705505,0.0272783152759075,0.1719476580619812,JST
1946,ESP,2.178995400217605,4.408685109666042,-0.0858151540160179,0.0302524305880069,0.3594751358032226,JST
1947,ESP,2.623060935923138,4.3763676148796495,0.1626156568527221,0.0268058460205793,0.4366329908370971,JST
1948,ESP,2.813467200714304,4.540295119182748,0.046996247023344,0.0260158907622098,-0.2906784117221832,JST
1949,ESP,2.965630400953257,4.626416840157298,-0.0438058674335479,0.0277168471366167,-0.1040574461221695,JST
1950,ESP,3.287347001653823,4.776119402985075,-0.0618063583970069,0.0301185566931962,0.0661356151103973,JST
1951,ESP,3.597263098829164,4.753416518122401,0.1334601640701294,0.030518103390932,0.1954616755247116,JST
1952,ESP,3.525839598942266,4.7511580947856045,0.0077590765431523,0.0324327647686004,0.096296302974224,JST
1953,ESP,3.5829784005844516,4.785833931562575,0.224649503827095,0.0329214744269847,0.0413147546350955,JST
1954,ESP,3.6270747013703857,4.81869654258523,0.1804903745651245,0.0296894125640392,0.2604258060455322,JST
1955,ESP,3.7730271997223017,4.844374470146542,0.0365501046180725,0.02864252589643,0.377144992351532,JST
1956,ESP,3.9941296992933193,4.866772113395791,0.0271813590079545,0.0284986421465873,0.4941006898880005,JST
1957,ESP,4.424534300072505,4.990019960079841,0.1060683876276016,0.0307437647134065,0.0921683609485626,JST
1958,ESP,5.015964901512411,5.0006250781347665,0.0399716310203075,0.0357647314667701,-0.118167832493782,JST
1959,ESP,5.382287700859441,5.0012503125781445,0.4530082046985626,0.0290662143379449,-0.0538770630955696,JST
1960,ESP,5.447371898986122,4.900159255175794,-0.2313524633646011,0.0382773801684379,0.02200866676867,JST
1961,ESP,5.5566418001133515,4.569339730408956,0.3300997912883758,0.0273905172944068,0.2837013304233551,JST
1962,ESP,5.873206501009648,4.351610095735422,0.1769192665815353,0.0286470502614974,0.2533000707626343,JST
1963,ESP,6.386227599227279,4.189797842254111,0.1337920725345611,0.0293097626417875,0.0889118835330009,JST
1964,ESP,6.832211402524457,4.23728813559322,0.1402118951082229,0.030959753319621,0.0159901604056358,JST
1965,ESP,7.641313603272756,4.184538131603724,0.1448513716459274,0.0374178849160671,0.0991163849830627,JST
1966,ESP,8.218578596649095,4.059677255658175,0.0865073502063751,0.0443140156567096,0.0783480703830719,JST
1967,ESP,8.742772499619223,3.86286817962337,0.0387361720204353,0.0563105084002018,0.0803629457950592,JST
1968,ESP,9.176652401674168,3.6032789838753265,0.0561339706182479,0.0665713250637054,0.2247946411371231,JST
1969,ESP,9.37403979739666,3.53356890459364,0.2142400294542312,0.0575900897383689,0.5244439840316772,JST
1970,ESP,9.91219980487014,3.624830086089715,0.089692011475563,0.0553757920861244,0.0952578261494636,JST
1971,ESP,10.726888004104415,3.369556060988965,0.1211879029870033,0.0523363724350929,0.0405203364789485,JST
1972,ESP,11.61233900379079,3.1680440771349865,0.1721067577600479,0.046943262219429,0.3475124835968017,JST
1973,ESP,12.940978998850644,4.786898066728948,0.1018180400133133,0.046496782451868,0.2975999414920807,JST
1974,ESP,14.878542003968924,4.421564004322625,0.1058645471930503,0.0479308143258094,0.0412130244076252,JST
1975,ESP,17.506959997412125,4.744478532293131,0.0306521169841289,0.0529218837618827,-0.1133933886885643,JST
1976,ESP,20.594397010128212,5.674551714886196,0.1096498221158981,0.0453931055963039,-0.1093562021851539,JST
1977,ESP,25.648258998217827,6.111111111111111,0.1930433213710785,0.0430136434733867,-0.2875902056694031,JST
1978,ESP,30.719811985589992,9.84724088172364,0.2529749870300293,0.0386602208018302,-0.1326136440038681,JST
1979,ESP,35.530664984532685,13.308194836048452,0.1955804377794265,0.0367072448134422,-0.0715109258890152,JST
1980,ESP,41.05751202042333,15.960833333333332,0.0673892125487327,0.0385367050766944,0.0096159568056464,JST
1981,ESP,47.0322059959101,15.811666666666667,0.0785903558135032,0.0404728204011917,0.4241999983787536,JST
1982,ESP,53.81134599319745,15.9875,0.0682861879467964,0.0424756631255149,7.298133823496755e-06,JST
1983,ESP,60.36330498761555,16.909166666666668,0.0735504701733589,0.0429565273225307,0.1242060288786888,JST
1984,ESP,67.17130799541991,16.5225,0.0935926213860511,0.0414869524538517,0.4670845866203308,JST
1985,ESP,73.09199900285351,13.3675,0.1128405258059501,0.039281565696001,0.3446483910083771,JST
1986,ESP,79.52105600491339,11.354166666666666,0.1799180805683136,0.0357954055070877,1.1325998306274414,JST
1987,ESP,83.69412400629847,12.813333333333333,0.2262890785932541,0.0307205710560083,0.479291558265686,JST
1988,ESP,87.74242799785586,11.744166666666668,0.2499616593122482,0.0261831115931272,0.1438328176736831,JST
1989,ESP,93.70222397299456,13.7025,0.2329981029033661,0.0231696516275405,0.126303568482399,JST
1990,ESP,100.00000000119084,14.6775,0.1562305390834808,0.0217325519770383,-0.117974504828453,JST
1991,ESP,105.93466000467863,12.360833333333334,0.1432258635759353,0.0207060445100069,0.0541034005582332,JST
1992,ESP,112.21102002980528,11.693333333333332,-0.0133968833833932,0.0227442421019077,-0.0808280557394027,JST
1993,ESP,117.33750003952984,10.211666666666668,-0.0040713832713663,0.0248354766517877,0.5609999895095825,JST
1994,ESP,122.87272997079562,9.995833333333334,0.0070478096604347,0.0260883029550313,-0.0920000001788139,JST
1995,ESP,128.61837995040491,11.27,0.0309445969760417,0.026681575924158,0.1570000052452087,JST
1996,ESP,133.193669939459,8.736666666666666,0.0258009899407625,0.0280849188566207,0.4289999902248382,JST
1997,ESP,135.6939412944318,6.401666666666666,0.0429664403200149,0.0286808721721172,0.4539999961853027,JST
1998,ESP,138.08577207508907,4.833333333333333,0.0487869158387184,0.0287266131490468,0.395000010728836,JST
1999,ESP,141.17325937022324,4.7275,0.0697406753897667,0.0278259832412004,0.1969999969005584,JST
2000,ESP,146.0919601372892,5.525833333333334,0.0750865489244461,0.0268628522753715,-0.1040000021457672,JST
2001,ESP,151.3359826273017,5.116666666666666,0.0947993099689483,0.0255779270082712,-0.0363768935203552,JST
2002,ESP,156.03371014859266,4.95916666666667,0.1693464666604995,0.0228017959743738,-0.2046543061733245,JST
2003,ESP,160.7764507384674,4.1241666666666665,0.1997265070676803,0.0198164600878953,0.3297805488109588,JST
2004,ESP,165.6624147283815,4.10333333333333,0.1830530762672424,0.0174369942396879,0.2308199852705001,JST
2005,ESP,171.2419891842005,3.3875,0.1456549912691116,0.0158705674111843,0.2578282356262207,JST
2006,ESP,177.2614640829975,3.785,0.1004429906606674,0.0150508284568786,0.4054714739322662,JST
2007,ESP,182.20267138435537,4.307499999999999,0.0547783747315406,0.0148925138637423,0.0916206240653991,JST
2008,ESP,189.6277816521101,4.36666666666667,0.0021031307987868,0.0154920872300863,-0.3694705069065094,JST
2009,ESP,189.08353273196053,3.97916666666667,-0.0756976604461669,0.0172760225832462,0.3560526967048645,JST
2010,ESP,192.48611150718125,4.25083333333333,-0.0264849308878183,0.0179430898278951,-0.1466962099075317,JST
2011,ESP,198.6385795631577,5.44,-0.0554687082767486,0.0191939417272806,-0.0877072066068649,JST
2012,ESP,203.49589887306388,5.845,-0.0873351320624351,0.021146323531866,0.0501080714166164,JST
2013,ESP,206.362412922423,4.56166666666666,-0.1557201296091079,0.0249999985098838,0.2988026738166809,JST
2014,ESP,206.0514135394804,2.72333333333333,0.003145863302052,0.0247502624988555,0.0908464267849922,JST
2015,ESP,205.02020505919705,1.73583333333333,0.035854909569025,0.0237784963101148,-0.0319905988872051,JST
2016,ESP,204.6048571990829,1.39333333333,0.0462261438369751,0.0227078907191753,0.0227000005543231,JST
2017,ESP,208.60692820589696,1.555,0.0619623661041259,0.0214711930602788,0.0971999987959861,JST
2018,ESP,212.10157916685728,1.4225,0.0672801733016967,0.0203791987150907,-0.1088000014424324,JST
2019,ESP,213.5849643815506,0.659166667,0.0517346858978271,0.0196843650192022,0.1623000055551529,JST
2020,ESP,212.8954460127897,0.375,0.0223709344863891,0.0194714423269033,-0.0980999991297721,JST
2021,ESP,219.29963878711084,0.3474166666666667,0.03295162906231752,0.018962765622591443,0.3192769432512965,extension
2022,ESP,237.55099062861788,2.1805,0.07481091110205718,0.017880047590475057,-0.12305995046665219,extension
2023,ESP,245.62836626586483,3.4761666666666664,0.041162391660243935,0.017525803874412386,0.20621503077457937,extension
2024,ESP,252.67110025522092,3.1485833333333333,0.07917226221595874,0.016596239457949673,0.24166793616665183,extension
2025,ESP,259.47480812130186,3.2175833333333332,0.1257843280299471,0.015097758835192838,0.09075463061088596,extension
'''
annual = pd.read_csv(io.StringIO(DATA_CSV))
annual.tail(8)

In [ ]:
# MODEL: presný výpočet oboch stratégií (src/model.py)
"""Kúpa s hypotékou vs. prenájom a investovanie: jeden výpočtový model pre obe úrovne.

Pravidlo férovosti: obe domácnosti míňajú v každom období rovnako. Kto má v danom období
lacnejšie bývanie, rozdiel investuje do rovnakého akciového indexu. Pre každý horizont h obe
domácnosti „speňažíme“ a porovnáme čistý majetok:

    kupujúci = predajná cena - náklady na predaj - zostatok hypotéky + jeho portfólio (po zdanení)
    nájomník = portfólio (po zdanení)

    výhoda = kupujúci - nájomník      (> 0  ->  kúpa vyhráva)

Všetko je vyjadrené na 1,0 kúpnej ceny, takže výsledok nezávisí od výšky ceny.
"""
from dataclasses import dataclass, replace
import numpy as np
import pandas as pd


@dataclass(frozen=True)
class Params:
    ltv: float = 0.80            # pomer úveru k cene (LTV)
    buy_cost: float = 0.10       # dane a poplatky pri kúpe (podiel ceny)
    sell_cost: float = 0.04      # makléri + miestne dane pri predaji (podiel ceny)
    upkeep: float = 0.012        # údržba + daň z nehnuteľnosti + poistenie, ročne, z aktuálnej hodnoty
    gross_yield: float = 0.05    # nájom v 1. roku / cena
    term_years: int = 30         # zadanie projektu: hypotéka štandardne na 30 rokov
    spread: float = 0.0          # variabilné úvery: Euribor + marža (určená pri čerpaní)
    invest_tax: float = 0.21     # daň zo zisku portfólia pri speňažení
    invest_tax_free_after: int = 0  # po koľkých obdobiach je zisk z portfólia bez dane (0 = nikdy)
    home_gain_tax: float = 0.0   # daň zo zisku z predaja bytu (v Španielsku sa predpokladá úľava na hlavné bývanie)
    home_gain_tax_free_after: int = 0  # po koľkých obdobiach je zisk z bytu oslobodený od dane (0 = nikdy)


SPAIN = Params()


def annuity(rate_per_period: float, n: int) -> float:
    """Splátka na 1,0 úveru."""
    if abs(rate_per_period) < 1e-12:
        return 1.0 / n
    return rate_per_period / (1 - (1 + rate_per_period) ** -n)


def simulate(price_g, rent_g, eq_r, mort_rate_path, p: Params, ppy: int, horizon: int, parts: bool = False):
    """Simuluje jeden dátum nákupu na `horizon` období.

    price_g, rent_g, eq_r : rast cien / nájmov / výnos investície za obdobie (desatinné čísla)
    mort_rate_path        : ROČNÁ sadzba hypotéky v každom období (pri fixnej konštantná)
    Vracia pole adv[1..horizon] = majetok kupujúceho - nájomníka (na 1,0 ceny).
    S parts=True vráti (kupujúci, nájomník, adv): čistý majetok každého scenára po speňažení v každom období.
    """
    n_total = p.term_years * ppy
    price = 1.0
    rent = p.gross_yield / ppy
    loan = p.ltv * 1.0
    bal = loan
    pay_n_left = n_total
    # počiatočná hotovosť: akontácia + náklady na kúpu idú do portfólia nájomníka
    renter = (1 - p.ltv) + p.buy_cost
    renter_contrib = renter
    buyer_port = 0.0
    buyer_contrib = 0.0
    adv = np.zeros(horizon)
    buyer_arr = np.zeros(horizon)
    renter_arr = np.zeros(horizon)
    for t in range(horizon):
        r_per = mort_rate_path[t] / ppy
        pay = annuity(r_per, pay_n_left) * bal if bal > 1e-12 else 0.0
        interest = bal * r_per
        bal = max(bal - (pay - interest), 0.0)
        pay_n_left = max(pay_n_left - 1, 1)
        buy_out = pay + p.upkeep / ppy * price
        rent_out = rent
        diff = buy_out - rent_out
        # najprv rast existujúcich portfólií, potom nový vklad na konci obdobia
        renter *= 1 + eq_r[t]
        buyer_port *= 1 + eq_r[t]
        if diff > 0:               # kúpa stojí viac hotovosti -> nájomník investuje rozdiel
            renter += diff
            renter_contrib += diff
        else:                      # prenájom stojí viac hotovosti -> rozdiel investuje kupujúci
            buyer_port += -diff
            buyer_contrib += -diff
        price *= 1 + price_g[t]
        rent *= 1 + rent_g[t]
        periods = t + 1
        # hodnoty po speňažení
        tax_free = p.invest_tax_free_after and periods > p.invest_tax_free_after
        tr = 0.0 if tax_free else p.invest_tax
        r_net = renter - tr * max(renter - renter_contrib, 0)
        b_net = buyer_port - tr * max(buyer_port - buyer_contrib, 0)
        home_tax_free = p.home_gain_tax_free_after and periods > p.home_gain_tax_free_after
        hg = 0.0 if home_tax_free else p.home_gain_tax
        gain = max(price - 1.0, 0)
        home_net = price * (1 - p.sell_cost) - hg * gain - bal
        adv[t] = home_net + b_net - r_net
        buyer_arr[t], renter_arr[t] = home_net + b_net, r_net
    return (buyer_arr, renter_arr, adv) if parts else adv


# ---------------------------------------------------------------- úroveň 1 (mesačne)
def tier1_grid(monthly: pd.DataFrame, rate_type: str, p: Params | None = None,
               max_years: int = 20, start_from: str = "2005-03", prices: str = "hp_ma"):
    """Spätný test pre každý mesiac nákupu. Riadky = mesiac nákupu, stĺpce = horizont v rokoch.

    prices: "hp_ma" = MIVAU valor tasado pre obec Málaga (od 2005-03), "hp_es" = celoštátny index BIS."""
    hp, hicp_rent, mir = monthly[prices], monthly["hicp_rent_es"], monthly["mir_es"]
    p = p or SPAIN
    eq = monthly["msci_eur_tr"]
    ph = hp.pct_change().shift(-1)           # rast z t na t+1
    rg = hicp_rent.pct_change().shift(-1)
    er = eq.pct_change().shift(-1)
    idx = monthly.index
    out = {}
    for i, m in enumerate(idx):
        if m < pd.Period(start_from, "M"):
            continue
        H = min(max_years * 12, len(idx) - i - 1)
        if H < 12:
            break
        rate0 = mir.iloc[i] / 100
        if rate_type == "fixed":
            path = np.full(H, rate0)
        else:
            spread = rate0 - monthly["euribor12m"].iloc[i] / 100
            path = np.clip(monthly["euribor12m"].iloc[i:i + H].values / 100 + spread, 0.005, None)
        adv = simulate(ph.iloc[i:i + H].values, rg.iloc[i:i + H].values, er.iloc[i:i + H].values,
                       path, p, 12, H)
        out[m] = {y: adv[y * 12 - 1] for y in range(1, max_years + 1) if y * 12 <= H}
    return pd.DataFrame(out).T


# ---------------------------------------------------------------- úroveň 2 (ročne, JST)
def tier2_run(jst: pd.DataFrame, year: int, horizon: int, iso: str = "ESP", p: Params | None = None,
              rate_spread: float = 0.015, parts: bool = False):
    """Jeden ročný nákup v roku `year` na `horizon` rokov; skutočné dáta JST (a predĺženie 2021–2025).

    Vracia (adv, cpi_pomer) alebo s parts=True (kupujúci, nájomník, adv, cpi_pomer); pomer CPI[k] =
    CPI(year+k) / CPI(year) slúži na prepočet na peniaze roku nákupu. Všetko na 1,0 ceny bytu."""
    d = jst[jst.iso == iso].set_index("year").sort_index()
    p = p or SPAIN
    cols = pd.concat([d["housing_capgain"], d["housing_rent_yd"], (d["ltrate"] / 100 + rate_spread).clip(lower=0.01), d["eq_tr"]],
                     axis=1, keys=["g", "ry", "rate", "eq"]).dropna()
    years = list(cols.index)
    i = years.index(year)
    H = horizon
    P = np.r_[1.0, np.cumprod(1 + cols["g"].values[i + 1:i + H + 1])]   # cena na začiatku roka k
    rent_path = cols["ry"].values[i:i + H + 1] * P                       # nájom_k = výnos_k * cena_k
    rg = rent_path[1:] / rent_path[:-1] - 1
    pp = replace(p, gross_yield=cols["ry"].values[i])
    out = simulate(cols["g"].values[i + 1:i + H + 1], rg, cols["eq"].values[i + 1:i + H + 1],
                   np.full(H, cols["rate"].values[i]), pp, 1, H, parts=parts)
    cpi = d["cpi"].reindex(years).values
    deflator = cpi[i + 1:i + H + 1] / cpi[i]
    return (*out, deflator) if parts else (out, deflator)


def tier2_grid(jst: pd.DataFrame, iso: str = "ESP", p: Params | None = None,
               start_from: int = 1950, max_years: int = 30, rate_spread: float = 0.015):
    """Ročný spätný test od `start_from` na národných dátach (reálne hodnoty, v peniazoch roku nákupu)."""
    d = jst[jst.iso == iso].set_index("year").sort_index()
    cols = d[["housing_capgain", "housing_rent_yd", "ltrate", "eq_tr"]].dropna()
    years = cols.index.values
    out = {}
    for i, y in enumerate(years):
        if y < start_from:
            continue
        H = min(max_years, len(years) - i - 1)
        if H < 5:
            break
        adv, defl = tier2_run(jst, int(y), H, iso, p, rate_spread)
        out[int(y)] = {h: adv[h - 1] / defl[h - 1] for h in range(1, H + 1)}
    return pd.DataFrame(out).T


def break_even(row: pd.Series):
    """Prvý horizont, od ktorého kúpa ostáva vpredu (None, ak nikdy)."""
    ok = row.dropna()
    wins = ok > 0
    for h in ok.index:
        if wins.loc[h:].all():
            return h
    return None


# ---------------------------------------------------------------- vektorizovaný model (pre sieť)
def simulate_batch(g, rg, eq, rate, ltv, buy_cost, sell_cost, upkeep, gross_yield, term_years,
                   invest_tax, inv_free_after, home_tax, home_free_after, years):
    """Rovnaká ekonomika ako `simulate`, mesačne, konštantné ročné sadzby, pre N scenárov naraz.

    Všetky argumenty sú polia tvaru (N,). `years` je horizont v rokoch; vracia výhodu
    (kupujúci - nájomník) na 1,0 ceny v danom horizonte.
    """
    N = len(g)
    gm = (1 + g) ** (1 / 12) - 1
    rgm = (1 + rg) ** (1 / 12) - 1
    em = (1 + eq) ** (1 / 12) - 1
    r = rate / 12
    n_left = (term_years * 12).astype(float)
    price = np.ones(N)
    rent = gross_yield / 12
    bal = ltv.copy()
    renter = (1 - ltv) + buy_cost
    renter_c = renter.copy()
    bport = np.zeros(N)
    bport_c = np.zeros(N)
    H = (years * 12).astype(int)
    result = np.zeros(N)
    for t in range(int(H.max())):
        # anuitná splátka zo zostatku
        pay = np.where(np.abs(r) < 1e-12, 1.0 / n_left, r / (1 - (1 + r) ** (-n_left))) * bal
        interest = bal * r
        bal = np.maximum(bal - (pay - interest), 0)
        n_left = np.maximum(n_left - 1, 1)
        diff = pay + upkeep / 12 * price - rent
        renter = renter * (1 + em)
        bport = bport * (1 + em)
        pos = diff > 0
        renter = renter + np.where(pos, diff, 0)
        renter_c = renter_c + np.where(pos, diff, 0)
        bport = bport + np.where(pos, 0, -diff)
        bport_c = bport_c + np.where(pos, 0, -diff)
        price = price * (1 + gm)
        rent = rent * (1 + rgm)
        done = (H == t + 1)
        if done.any():
            per = t + 1
            tr = np.where((inv_free_after > 0) & (per > inv_free_after), 0.0, invest_tax)
            hg = np.where((home_free_after > 0) & (per > home_free_after), 0.0, home_tax)
            r_net = renter - tr * np.maximum(renter - renter_c, 0)
            b_net = bport - tr * np.maximum(bport - bport_c, 0)
            home_net = price * (1 - sell_cost) - hg * np.maximum(price - 1, 0) - bal
            result = np.where(done, home_net + b_net - r_net, result)
    return result

In [ ]:
# VÁHY NEURÓNOVEJ SIETE z webu (natrénované na simulovaných scenároch, src/train_nn.py)
NN_JSON = '{"features":["years","ltv","gross_yield","buy_cost","house_growth","rent_growth","equity_return","mortgage_rate"],"gapScale":3.0,"ranges":{"gross_yield":[0.01,0.07],"house_growth":[-0.05,0.2],"rent_growth":[0.0,0.2],"equity_return":[0.0,0.2],"mortgage_rate":[0.01,0.2]},"mu":[16.004541666666668,0.7000642787500966,0.039996472249189764,0.09241416491511936,0.07472093141010402,0.06034301834358487,0.10024820415184328,0.10497067353738924],"sd":[8.36579271338532,0.11551481014438747,0.01730134902246168,0.02164879408488096,0.07210365862465948,0.04931219108275712,0.05777096532357296,0.05486655414813875],"layers":[{"W":[[0.045953,0.428036,-0.247129,0.332133,0.540539,-0.560679,-0.672307,0.446637,-0.888594,0.265792,0.086206,-0.224591,0.280191,0.156447,-1.971864,0.596071,-0.725732,-0.09322,0.098377,-0.29649,-0.114943,-0.10439,-0.006218,0.206542,0.250518,-1.229817,0.235148,-0.828056,0.282712,-0.231871,0.295732,0.499609,-0.497766,0.281357,0.70064,0.392879,-1.091017,-0.190813,0.733209,0.613929,0.126454,0.295111,-0.811005,-0.087817,-1.182599,0.6,0.259293,0.759584,-0.798055,0.436867,0.904794,-0.426528,0.526265,0.099951,0.686963,0.167519,0.211712,0.572775,-0.238965,-0.009891,0.686815,-0.129571,1.096939,-0.502692],[-0.07366,0.356243,-0.06661,-0.3171,-0.164377,-0.387643,0.074229,0.020706,0.348093,0.307227,0.5661,0.306866,0.013185,0.434548,-0.007725,-0.270209,-0.332437,0.562961,-0.202996,0.123496,0.171464,-0.064195,0.044603,-0.050153,0.08585,0.136603,0.851913,0.282582,-0.048943,-0.420592,-0.582088,0.219369,-0.169862,-0.116442,0.947501,-0.898625,0.12664,0.293049,0.119984,-0.438754,0.389737,-0.468967,-0.350423,-0.726139,-0.154843,-0.248518,-0.362941,-0.048139,-0.095791,0.005745,0.374943,-0.306548,0.391517,0.173885,-0.253239,0.133085,-0.37709,0.651283,0.229865,-0.402374,0.556136,-0.129263,-0.149981,0.5814],[-0.046637,-0.155354,-0.229658,1.081953,-0.186537,0.165126,0.071727,-0.53543,-0.155135,-0.351754,0.790016,-0.446934,-0.208408,-0.300568,-0.09355,-0.443683,0.06395,0.010503,0.380518,0.624468,0.125592,0.136285,0.093421,-0.374449,0.59833,0.433866,-1.171176,0.09595,0.604499,0.025631,0.008259,0.368385,-0.081569,-0.089499,0.248997,0.180753,0.287197,0.183947,0.071558,0.222614,0.041519,-0.208993,0.170855,-0.24445,-0.664864,-0.080529,-1.18464,-0.447036,-0.0132,0.150813,0.086339,-0.1291,0.464557,0.372147,0.125784,0.632497,-0.412711,0.001517,0.359045,0.247361,0.259787,0.543633,-0.373733,-0.168914],[-1.101958,-0.034004,0.72147,-0.182912,0.213576,-0.607195,-0.16036,0.180098,0.579424,-0.612314,-0.30928,-0.021149,0.345441,-0.779841,0.007198,0.029192,-0.60527,-0.366126,0.167368,-0.636536,0.1312,-0.008609,0.040396,-0.272779,-0.34431,-0.077488,-0.348627,0.083748,-0.121359,-1.115593,-0.41002,-0.258655,0.382416,-0.116363,-0.18703,-0.094773,-0.09082,-0.065774,-0.310903,0.24367,1.140129,0.385221,-0.059339,-0.613625,0.211819,0.310975,0.130647,0.016315,-0.036082,-0.031273,-0.54942,-0.215861,0.598579,-0.08837,0.62342,-0.657866,-0.466459,0.090116,-0.881386,0.651987,0.207821,0.090917,-0.413911,0.027614],[-1.149096,-0.301449,-0.790566,1.155719,0.321189,-0.513287,-0.765916,-0.269896,-0.070464,-0.577374,-0.061693,-0.585973,-0.204847,-0.438132,-0.44293,-0.565446,-0.281644,-1.011951,0.160798,0.186615,0.535875,1.492035,-0.568371,1.26041,0.390833,0.62751,-0.536424,-1.372966,-0.563766,-1.048221,-0.261499,1.249967,-0.699893,-1.17478,-0.472769,-0.000577,-0.108187,-0.78457,-0.1529,0.991559,0.657216,0.47973,0.180748,0.495732,0.130876,-1.168215,-0.546271,-0.911944,-0.242052,1.680756,0.975078,-0.15105,-0.264909,-0.70871,0.265388,-0.970015,-0.471514,0.480706,0.334364,0.537428,0.633122,-0.224156,-0.449036,0.041937],[-0.058543,-0.400949,0.566696,-0.338579,0.177005,-0.70443,0.105628,-0.191092,-0.603391,-0.629858,0.999942,-0.018624,-0.229632,0.63869,-0.052576,-0.431609,0.389227,1.064019,-0.579578,-0.215544,0.662794,-0.250367,-0.706291,-0.624514,0.069334,0.334589,-0.000988,0.195063,0.999039,0.882087,0.467127,-0.109845,-0.315592,0.02474,0.328266,-0.050464,-0.299246,-1.196256,0.008873,0.157526,-1.058984,-0.767846,0.296131,-0.529155,0.296625,-0.171838,-0.299768,-0.389327,-0.163217,0.371067,0.15797,0.13639,0.185081,-1.19519,0.190258,0.277307,0.954394,0.562783,-0.130335,0.600155,0.20287,0.40493,-0.102032,-0.571467],[0.053058,-0.124547,0.056644,-0.289143,-0.849855,0.413352,-0.225996,0.575449,-0.353366,-0.159014,-0.104603,-0.705947,0.71605,0.443435,-0.011681,-0.263411,0.722667,0.219383,-0.905607,-0.198968,-1.009868,-0.618744,0.749244,-0.255186,0.658897,-0.527921,0.395573,0.501774,-0.30964,-0.13305,-0.249319,-0.588174,0.501207,0.935972,-0.790507,0.671699,0.875934,0.004439,0.201363,-0.042072,-0.496961,0.044732,-0.36492,-0.224224,0.120196,0.515348,0.857991,1.199017,-0.293025,-1.31589,-0.422148,-1.066532,0.357196,1.317,-0.823856,0.405098,-0.53471,0.438521,0.005367,0.038626,0.493085,0.339486,1.185501,0.621752],[0.044569,0.74939,-0.311963,-0.039387,1.130833,-0.642763,0.743438,0.624855,-0.023436,0.669848,-0.83528,0.271376,-0.087069,0.346239,0.145626,0.227475,0.661926,-0.379654,-0.08603,-0.819591,-0.556991,-0.050016,0.354672,-0.041455,-0.384896,-0.42913,-0.207489,0.754314,-0.191753,0.098437,-0.976489,-0.178022,0.33762,0.245144,-0.307622,-0.306337,-0.074774,0.402316,-0.911065,-0.093225,0.232708,-0.498919,0.131267,0.183652,0.628849,0.071468,-0.586228,0.311083,-0.013674,-0.21578,-0.225743,0.051659,-0.037946,0.724672,-0.300316,0.355541,0.814943,-0.232238,0.587178,-0.625394,-0.276882,-0.155828,0.194498,-0.499695]],"b":[0.058719,0.316807,-0.054638,0.390701,-0.525704,-0.363685,0.139432,0.602423,-0.34645,-0.052554,-0.437385,0.509527,-0.131808,0.000196,-0.691427,0.122332,0.071506,-0.396359,-0.283231,-0.104204,0.061208,0.029939,0.508285,0.161123,-0.383168,-0.708589,-0.261918,0.182052,-0.099357,-0.396315,-0.038344,0.170601,0.120262,-0.010283,-0.581463,-0.700686,-0.580498,0.023208,-0.923013,-0.229237,-0.299563,-0.052707,-0.876703,0.145621,-0.528295,0.299334,0.069481,-0.985919,1.093774,-0.442007,0.009571,-0.445786,-0.104871,-0.28959,0.105687,-0.156103,-0.299825,-0.22198,-0.806925,-0.254522,-0.213318,0.018819,-0.053421,0.13526]},{"W":[[-0.365995,-0.116787,-0.027569,0.437462,-1.600542,-1.928429,-0.038843,-0.04412,0.224324,-0.675756,0.357528,0.191445,0.269674,-0.156804,-0.017741,0.230402,0.317334,0.042862,0.073918,0.06137,0.152479,0.216507,0.328322,0.052719,-0.152072,0.153413,0.035127,-0.26668,0.268262,-0.051981,0.054753,0.169375,0.076509,0.434836,-0.339654,-0.260527,0.033646,0.109223,-0.772055,-0.222665,0.2242,0.145447,0.548799,-0.041481,-0.137393,-0.014631,0.342642,-0.524036,0.244408,-0.304333,-1.276888,-0.216149,0.000422,0.099537,-0.241864,0.161598,-0.088041,-0.227492,-0.089961,-0.262919,0.473973,-0.236304,-0.285187,-0.241905],[-0.059494,-0.372752,-0.239385,0.327797,0.154129,-0.00718,-0.103245,-0.162216,0.615934,0.366231,0.160839,-0.095854,-0.101194,-0.455429,-0.190278,-0.045227,0.180882,0.26819,-0.605672,0.15095,0.15511,-0.264199,0.23845,-0.624232,0.113094,0.023111,-0.035642,0.502572,-0.007419,0.273558,-0.11251,-0.364608,-0.909739,-0.422198,-0.019199,-0.916192,-0.257544,0.302652,-0.599113,0.026341,-0.173262,-0.723882,-0.328577,0.179988,0.132558,0.031272,-0.068691,-0.583235,0.107349,-0.242765,0.373937,0.114132,0.157141,0.056711,0.177514,0.244345,-0.056291,0.116779,-0.27272,-0.129437,0.105936,-0.211788,-0.509656,-0.921698],[-0.1334,0.055649,0.07971,-0.622,0.407399,0.097834,-0.047441,-0.307079,0.103339,0.188488,0.169644,-0.940592,-0.313016,-0.12202,0.451116,0.332689,-0.606715,0.059985,-0.064355,0.113862,0.162213,0.139807,0.479295,-0.711726,0.597186,-0.036301,-0.179617,0.324205,0.046965,0.101312,0.013065,0.602605,0.385434,0.206855,-0.225323,-0.369937,0.125687,-0.658931,0.311826,-0.027365,0.005687,0.428709,0.126311,0.069174,0.15926,-0.485631,0.253102,-0.347228,0.011208,-0.419202,-0.054526,0.090707,0.108554,0.173495,0.651574,-0.089373,-0.045908,0.034239,0.079557,-0.049589,-0.963985,0.138196,-0.710682,0.003143],[-0.360639,0.050705,-0.110567,-0.040565,0.152948,-0.418003,0.278299,-0.509917,-0.263987,-0.089163,-0.461575,0.332082,-0.067036,0.197733,0.523748,-0.309648,0.279092,0.049248,-0.15425,0.09797,-0.228078,-0.209355,-0.317592,0.254289,0.115049,-0.135488,-0.009845,0.137755,-0.690144,0.045509,0.017502,-0.083557,0.002138,0.362326,0.225168,-0.018513,0.110847,0.204134,0.392099,-0.559194,0.192732,0.23132,-0.84193,0.049118,0.366413,-0.100545,-0.022162,-0.080711,0.307503,-0.146374,-0.171339,-0.061392,-0.139631,0.713399,0.148183,0.404904,-0.104315,-0.204034,0.180006,-0.76642,-0.068615,0.40817,0.038269,-0.009561],[0.087555,0.428857,0.214073,0.381926,-0.281872,0.088937,0.245555,-3.002624,-0.006335,-0.00472,0.171321,0.141982,-0.503421,0.179367,0.115288,-0.083825,0.257879,-0.147078,0.156604,0.070822,-0.331633,0.237984,0.658679,-0.728932,-0.271773,-0.182902,-0.058665,0.198544,0.167213,0.045128,0.457779,0.103546,-1.829384,-0.25088,0.385133,-0.466805,0.229709,0.157673,-0.146352,0.026532,0.481842,0.530277,-0.06181,0.013986,-0.486434,-0.100429,-0.083406,-0.187161,0.38675,0.366403,0.001684,0.35161,-0.026961,-0.035689,-0.247806,0.008254,-0.43783,0.164369,0.21944,0.104881,0.149135,0.339073,0.48523,0.230774],[-0.562529,-0.042534,-0.116955,-0.676348,-1.24554,-3.188506,-0.045446,-0.289202,0.471734,0.367041,-0.166083,0.061879,-0.058621,-0.170768,-0.290451,-0.196607,0.450163,-0.192226,-0.2005,0.072626,0.726432,0.163698,0.070531,0.313769,0.104029,0.040982,0.121337,-0.609168,0.297978,-0.013934,-0.028959,0.215781,0.081395,0.632226,0.222852,0.101425,-0.316222,0.04259,-0.362774,0.137671,-0.008422,-0.008781,0.175322,0.038438,0.058515,0.389716,0.510313,-0.290119,0.021215,0.120291,-0.180755,0.082681,0.037948,-0.502339,0.36552,0.158573,0.485359,0.22339,-0.302232,0.630559,-0.358891,-0.022605,0.288689,0.153977],[0.050187,-0.385081,0.111842,0.092296,-3.231423,0.009559,-0.164711,-0.603078,-0.143263,-0.002234,0.118762,0.07235,-0.380772,-0.154241,0.337183,0.77951,-0.055511,-0.126245,0.013916,-0.160117,-0.570682,-0.746284,-0.471011,0.419952,0.275928,0.256119,0.001185,-0.606244,-0.482571,-0.025175,-0.62859,-0.339259,-1.027544,-0.438028,-0.023427,0.344253,-0.277883,0.155616,-0.083043,-0.038371,-0.086027,-0.285041,-0.189588,-0.041036,0.110857,-0.321685,-0.072133,-0.165815,-0.279437,0.368621,-0.126409,0.586497,-0.2395,0.229829,-0.769565,-0.437274,0.262489,0.306264,-0.151426,0.414183,-0.026814,-0.078438,0.486149,0.550185],[-0.140871,-0.192537,-0.116495,0.033836,-0.224129,0.089718,0.150354,-0.093467,0.039568,-0.225685,0.456647,0.006862,0.043966,0.093616,-0.565557,-0.425713,0.49265,0.553888,-0.247528,0.383236,-0.329855,0.378512,0.297806,-2.074093,-0.259371,0.219552,-0.177422,-0.061888,-0.230609,0.313606,-0.173436,-0.869999,-0.190813,-0.332662,-0.111478,0.100707,0.129668,-0.374606,0.299274,-0.26886,0.270484,0.276795,0.024409,0.225146,0.117309,-0.091651,0.010466,-0.284173,-0.265308,-0.093764,-0.725144,0.1109,-0.212672,-0.044197,-0.147857,0.149588,-0.191411,0.282041,0.031485,-0.32947,0.320067,0.17122,-0.426129,0.360036],[0.43449,-0.155862,-0.344262,-0.171075,0.143596,0.15604,-0.235893,-0.217726,0.097149,0.071907,-0.029299,-0.176045,0.152447,-0.095272,0.019627,0.100973,-1.538961,-0.318687,0.314035,-0.093849,-0.213388,-0.763645,0.27008,0.417763,-0.251809,-0.144755,-0.006931,-0.617971,0.533689,-0.19536,-0.103235,0.466486,0.004023,0.164778,0.918159,0.178902,0.132457,0.322885,-0.083503,0.288284,-0.323701,0.339295,-0.594095,-0.0983,0.205821,0.312473,0.073233,0.13346,-0.716172,-0.3908,0.131518,-0.036503,-0.215817,-0.152286,0.114505,-0.181708,0.173102,0.015157,0.211354,0.276063,0.02918,0.072798,-0.080278,-0.079458],[0.731757,0.003568,0.019952,0.236841,0.058761,-0.013915,-0.130409,-0.043287,0.197555,-0.700197,-0.064699,-0.140772,0.410586,-0.12679,-0.016446,0.367241,0.227321,0.056431,0.14626,0.092236,-0.041054,-0.308721,-0.235163,-0.72524,0.034213,0.48822,-0.19684,-0.141139,0.596816,0.138488,-0.158733,0.093313,-0.310766,-0.438868,0.642549,-0.160314,-0.387937,0.085862,0.739926,0.228716,-0.603447,-0.209934,-0.462925,0.119725,-0.100489,0.188362,0.126722,-0.289393,-0.873259,0.170322,-0.424416,0.025544,0.224652,-0.111095,-0.202587,-0.034584,0.386997,-0.022136,-0.171551,0.165855,-0.003513,-0.235986,-0.378873,0.257308],[0.042871,0.100425,-0.035115,0.184094,-0.485119,-1.758968,0.056598,0.279897,0.158117,-0.098552,0.062976,-0.181935,-0.026389,-0.240326,0.43523,-0.101454,0.074483,-0.193726,0.223531,-0.198501,0.405839,0.434016,0.220328,0.414634,0.098412,0.205308,-0.065607,0.107268,-0.080682,-0.427173,0.063088,0.073527,0.308158,0.056426,0.19869,0.263501,0.176969,0.052615,-0.059681,0.247091,0.095698,0.041406,0.078671,-0.350948,-0.625568,-0.644007,-0.129753,0.05641,0.20763,-2.351143,0.133844,0.056604,0.018842,0.054998,-0.277938,0.348723,-0.120687,0.021843,0.281146,0.006154,-0.837751,0.191893,0.268537,0.166702],[0.158522,-0.624405,-0.324037,0.199624,0.309071,0.127149,0.090574,-0.205665,0.257553,-0.595411,0.029143,-0.484107,0.475273,-0.077311,-0.288171,-0.016789,-0.397417,-0.033582,-0.027179,0.06601,-0.535243,-0.279187,0.280266,-0.051488,-0.393142,-0.10786,-0.143516,0.376126,-0.294051,0.024621,-0.611392,0.132612,0.034988,-0.484475,-0.294036,0.416988,-0.628941,-0.133509,-0.232691,-0.403165,-0.006689,0.07728,0.36836,0.018787,0.196094,0.228298,0.1583,-0.00432,-0.036838,0.310334,-0.91662,-0.33882,-0.016478,-0.136145,-0.175077,-0.190821,-0.168149,0.153202,-0.625942,0.12326,-0.104155,-0.129874,-0.103596,0.394912],[-0.259027,0.068238,-0.090959,-0.213255,0.289716,-0.373872,-0.093758,0.360556,-0.082946,0.299563,-0.011133,0.323071,0.084712,-0.18466,0.156764,-0.390552,0.012734,0.107977,0.01437,0.100916,0.106264,-0.600113,-1.666823,-0.005989,0.165797,0.133847,0.055171,0.30889,-0.166255,0.184026,0.003418,-0.059237,-0.042683,0.602935,0.212808,0.100578,-0.283185,-0.210677,-0.031174,-0.109073,0.237363,0.129247,0.024815,0.178253,0.107359,-0.247422,-0.526487,-0.037578,-0.165701,-0.04459,-0.194877,-0.16921,-0.0496,-0.240395,-0.025766,-0.130824,-0.11539,0.300721,-0.271456,0.108716,-0.265688,-0.12645,0.150576,-0.097435],[0.418775,-0.162892,-0.071022,-1.034721,-0.062443,0.235,0.057967,-0.482491,0.153997,0.34661,-0.079806,-0.255904,0.255141,0.214224,0.102118,-0.205062,0.040623,0.187371,-0.115677,0.049828,-0.165293,-0.223582,0.39857,-0.271252,-0.737573,0.163595,-0.116305,0.507642,-0.207433,0.098288,-0.053981,-0.065773,-0.685756,-0.138174,-0.527782,0.034693,-0.115739,0.056337,0.180833,-0.226729,-0.052783,-0.488835,-0.078391,0.083731,-0.028503,0.421763,-0.196078,0.344725,-0.08903,-0.296902,-0.748896,-0.279278,0.30975,0.017147,-0.370249,-0.082375,-0.175391,-0.228741,-0.157275,-0.501742,0.66152,0.487017,-0.044021,-0.306365],[-1.121145,0.864865,-0.871674,-0.472686,-0.309296,-0.439993,-0.823236,-0.124543,0.418314,0.44125,0.028385,0.09261,-0.049566,-0.081212,-0.40962,-0.143927,0.258308,-0.443796,0.230274,-0.789947,0.101279,-0.335135,-0.024825,-0.041828,0.241761,-0.189708,-0.905445,-0.27626,-0.03686,-0.880898,0.443483,-0.053761,0.068769,-0.787722,-1.13669,0.23494,-0.328701,0.160707,-0.302553,0.539158,-0.29521,-0.076862,-0.449721,-0.608729,0.391315,-0.086917,0.723584,-0.038109,-7.40895,-0.350714,0.407808,0.504957,0.249997,-0.03238,-0.817347,-0.420966,-0.302138,-0.248568,0.099945,-0.183944,-1.067418,-0.289181,0.011835,-1.075467],[0.14571,0.070471,0.123758,-0.098909,-0.48463,0.526237,-0.023717,-1.28081,0.174153,-0.105758,0.076659,0.370615,-0.078554,-0.620934,-0.458762,0.088197,0.270844,0.164464,0.147919,0.020646,0.212776,0.383573,-0.648956,0.373813,0.59935,0.040139,-0.083894,0.859013,-0.314186,-0.218538,0.228532,-0.261104,-0.105237,-0.135835,-0.136911,-0.274561,0.38389,0.066818,-0.5974,0.178128,0.374495,0.041327,-0.074614,-0.233613,0.019844,-0.043473,0.310482,-0.30546,-0.328709,0.024403,-0.393271,-0.273269,-0.224141,-0.285037,-0.424373,0.020886,0.145687,-0.18589,0.104812,0.285294,-0.075759,0.410393,0.365786,0.517058],[0.143279,-0.092154,-0.004978,-0.322393,-2.676838,0.069498,0.009405,-0.046201,0.094815,-0.065658,0.112082,0.20961,-0.038511,0.282324,0.039436,0.171228,0.027877,-0.152061,0.156566,-0.055988,0.518809,-0.445538,-1.597541,0.016503,0.199755,-0.031685,0.04046,0.483488,0.085774,-0.397041,-0.126099,0.002337,0.402922,-0.545272,-0.527771,0.297346,-0.15562,-0.05081,-0.426473,0.043981,0.044246,0.329942,0.118985,-0.394767,0.679196,-0.901076,0.088044,0.207194,-0.129483,0.19228,-0.076878,-0.207308,-0.024278,-0.130147,0.424547,-0.119568,-0.08005,-0.003197,-0.175327,-0.222428,-0.073046,0.007808,0.096221,-0.442924],[0.466661,0.046523,-0.030824,-1.011347,-0.286862,-0.499564,0.00076,0.478872,-0.067661,0.294404,-0.33854,0.237418,0.260281,-0.450384,0.851798,-0.730304,0.118732,-0.29263,0.177558,0.081298,0.246229,0.811162,-0.209154,0.039431,-0.046234,0.039406,0.072304,-0.55977,-0.079769,-0.093859,-0.007485,0.202162,0.069054,-0.120144,0.331716,0.000241,0.114035,-0.051604,0.115612,0.293994,0.429684,-0.130283,0.569118,-0.051471,-0.390818,0.009692,-0.291552,-0.105287,-0.303282,0.195894,0.185556,0.125789,-0.20469,-0.210385,-0.185771,0.175308,0.096334,0.182639,0.050371,-0.512949,0.318997,-0.128452,-0.005679,-0.952039],[0.323259,-0.318295,-0.012523,-0.428208,-0.723937,-0.22643,-0.165278,-1.596809,0.211687,0.028608,-0.455939,-0.429208,0.280449,0.29856,0.020808,-0.202044,-0.269024,-0.250686,-0.085589,-0.156026,-0.344735,0.046377,-0.30909,-0.657265,-0.971153,0.005268,-0.078671,0.031484,-2.799797,-0.179954,-0.472932,0.151017,0.357059,0.110187,0.050157,0.251567,-0.250243,-0.219644,0.078218,0.29572,0.340809,0.126169,0.075384,-0.128853,-0.309286,-0.067616,-0.525791,-0.035974,-0.111826,0.014427,0.230466,0.247847,0.238915,-0.222554,-0.326572,-0.534417,0.163314,0.045757,-0.204396,0.184974,0.152936,-0.048161,-0.312049,-0.116783],[-0.193655,0.027437,0.263646,-0.791252,-1.547659,-1.663401,-0.194303,0.06851,-0.331879,-1.22022,-0.169507,0.113165,-0.132634,-0.000264,-0.383274,0.362651,0.641028,0.234274,-0.114214,-0.046814,-0.158499,-0.194009,0.132177,-0.009908,0.022102,-0.209814,-0.039182,0.309027,-0.182292,-0.182237,-0.05303,0.185024,0.177764,-0.088919,0.090198,-0.555845,-0.001288,0.10539,0.098724,0.12765,-0.351974,0.084266,0.162816,-0.227725,-0.443252,-0.432844,-0.495835,-0.25034,0.077876,0.41726,0.117554,-0.116423,0.014033,-0.178433,-0.079774,-0.222064,0.552272,-0.341521,0.055049,-0.211314,-1.024339,-0.006735,0.59298,0.287068],[-1.153805,0.055439,0.211728,-0.539636,0.686955,0.254446,0.146837,-0.395828,-0.104835,-0.274738,-0.782464,-0.953583,-0.517085,-0.055016,-0.055231,-0.320566,-0.28367,-0.375579,0.142561,0.047753,0.423347,0.177853,-0.266819,0.38557,-0.678227,-0.12869,0.244117,0.285808,0.285539,-0.110911,-0.097023,0.389062,-0.104443,0.265261,0.247573,0.336747,-0.123615,0.201408,0.186006,0.112478,0.129515,0.21124,0.506295,-0.121544,0.085321,-0.215861,0.019324,0.029314,0.068724,0.034155,0.427832,0.060316,-0.021997,0.252925,0.196138,0.296326,-0.962442,-0.137563,-0.141487,0.235337,0.504579,-0.286546,-0.321928,0.453485],[-0.906498,0.287505,-0.021895,0.615714,-0.49707,-0.110383,0.087772,-0.351312,0.140072,0.158916,-0.757946,-0.062133,-0.521941,0.129838,-0.110424,-0.465433,-0.083289,-0.723304,0.125705,-0.935536,-0.13484,-0.01593,0.197764,-0.019508,-0.108418,-0.578531,-0.148668,-0.000309,-0.353366,-2.026891,0.364143,0.151778,0.15614,-0.167347,0.230857,-0.658573,0.554759,0.004441,0.359987,0.055238,-0.115626,0.291622,-0.026602,-1.599903,-0.351495,-0.527248,-0.045864,0.25634,0.131771,0.099081,-0.03353,-0.083115,-0.085046,-0.111237,0.315916,-0.267588,-1.936679,-0.505146,0.567533,0.389392,-0.09419,0.292488,-0.185082,-2.091483],[0.070887,-0.281218,-0.015121,-0.036426,-0.140707,0.332385,-0.040938,0.049587,-0.11806,-0.026823,0.168687,0.315183,-0.161049,0.130161,-0.418399,0.084531,0.294898,0.125035,-0.457054,0.407043,0.024772,-0.140164,0.405306,-0.10292,0.165955,0.508706,0.034243,-0.476285,-0.117597,0.241416,-0.196247,-0.409972,0.024938,0.147978,-0.033499,-0.210992,-0.20197,-0.256869,-0.110592,-0.303386,-0.06432,-0.012922,-0.679095,0.229648,0.084185,-0.111347,-0.473378,0.199985,0.023832,0.089456,-0.898296,0.087765,0.238993,0.201294,0.25944,0.424162,-0.34798,0.315014,-0.546028,0.01129,0.20341,0.161215,-0.085997,-0.009699],[-0.046383,0.678297,0.344633,0.114322,-0.102176,-0.014555,0.362815,-0.234163,0.064433,0.609615,-0.319933,-0.206408,0.186047,0.367921,-0.147054,-0.094806,-0.146905,-0.138396,-0.086635,0.009923,-0.243664,0.104578,-0.64597,-0.091876,-0.325505,-0.232174,0.064664,0.088089,-0.076826,-0.165404,0.6212,-0.098074,0.342538,0.43414,0.51266,-0.157444,0.459673,0.239705,0.161204,0.1078,0.056532,-0.591689,0.047299,-0.092875,-0.502048,-0.584916,0.169169,0.828869,-0.006694,0.022343,-0.613615,0.235853,0.138042,0.331557,0.183976,0.140733,0.223367,-0.063994,0.387139,0.064174,-0.580325,-0.528948,0.294831,-0.794868],[-0.08631,0.006583,0.044167,-0.018893,-0.267709,-0.043348,0.098244,0.109881,0.057208,0.461821,0.176762,-0.288939,-0.200127,-0.278878,-0.665558,0.053422,0.012427,-0.030382,0.064404,0.093611,0.087411,-0.124495,-1.734874,-0.418887,-0.070679,-0.301088,0.139991,0.024182,0.314922,0.176957,-0.011512,-0.100344,0.104786,0.551998,0.227251,0.371237,-0.117179,0.106186,-0.460435,0.073133,0.038925,0.283991,0.124487,0.190679,-0.342761,0.011459,0.373553,-0.038109,-0.004459,-0.338553,-0.284045,-0.003589,0.318826,-0.541599,0.418503,0.084714,0.24084,0.08217,-0.233271,0.225827,-1.778219,0.10441,0.378298,0.068679],[-0.647292,-0.699413,-0.34571,0.280443,-0.836558,-0.123622,0.000503,-0.883216,0.410301,0.05559,-0.088492,0.102214,-0.318223,-0.179194,-0.340794,-0.130461,-0.321023,0.897846,-0.34111,0.352592,-0.297162,0.668881,0.289766,-0.198461,-0.014133,-0.649554,0.087781,0.232667,-0.878299,0.079802,-0.562767,-0.308461,-0.066403,0.243498,-0.316998,0.141883,0.011321,0.377196,0.205001,0.402383,0.096462,0.463268,-0.158418,-0.266312,-0.541492,0.155184,0.438513,0.294251,-0.63998,-0.836488,0.334206,0.395275,0.183415,-0.161856,0.008626,-1.629281,-0.172445,0.176355,-0.310675,-0.122667,0.476306,-0.109784,-0.719691,0.096579],[0.14155,0.012452,0.00594,-0.048423,0.032793,-1.220915,-0.182898,-0.10474,-0.010227,-0.140655,0.559759,0.060842,-0.068762,-0.631513,-0.169888,0.502409,-0.354241,-0.021156,0.263237,-0.021217,0.029786,-0.371039,0.039836,0.26003,0.384084,0.06189,0.02304,0.068494,-0.163377,0.060341,0.034059,0.345958,0.113683,-0.133929,0.01816,-0.219756,-0.097474,-0.088377,-0.12594,0.272716,-0.059419,-0.14826,0.398986,0.063266,0.738186,0.193304,0.029424,0.53407,0.057795,-0.429756,0.249681,0.162952,-0.025472,0.036136,0.465737,-0.015593,0.336892,0.157614,-0.128148,0.039509,-2.534968,-0.682741,0.371597,-0.086832],[-0.149982,-0.228065,-0.072732,0.19403,0.29889,0.254042,-0.387877,0.567431,-0.038512,0.487499,-0.022685,-0.120118,0.03195,0.037439,-0.060809,0.127034,-0.681684,0.118771,0.084892,0.079368,-0.017328,-0.122161,-0.137569,0.278914,0.317397,-0.109245,0.130178,0.066846,-0.540881,0.099375,-0.317659,-0.643226,-1.975486,-0.269883,0.306716,0.295272,-0.035325,0.208629,0.065221,-0.158618,0.175617,-0.078909,-0.094592,0.039,-0.178649,0.386838,0.125599,-0.124781,-0.986622,0.182326,0.048036,-0.049387,-0.058311,0.063396,-0.146283,-0.958667,0.139791,0.248146,-0.185479,0.301035,-0.518331,0.027512,-0.37953,0.118816],[0.192042,0.560405,-0.004307,-0.091445,-1.198013,-1.408353,0.022179,-0.175418,-0.052328,-1.07649,-0.53962,-0.016613,-0.856656,0.143105,0.285861,0.029065,-0.051395,-0.519163,0.221159,-0.68292,-0.23204,0.908532,0.404391,-0.274379,0.427719,0.235212,-0.281974,-0.491706,0.163931,-1.220257,0.730657,-0.18011,0.422627,0.270396,0.700697,-0.403644,0.703791,-0.03156,-0.098965,-0.256477,-0.157751,-0.050221,-0.306482,-1.049141,0.294091,-0.140143,0.473116,0.963848,-0.1309,0.138819,-0.215021,-0.104788,0.02165,0.146294,-0.376558,-0.747799,-0.575042,-0.030151,0.75412,-0.323389,0.120402,0.812801,0.577528,-0.161691],[-0.575716,0.015177,0.081028,-0.382725,-1.148846,-0.064322,0.083095,0.547755,-0.112262,-0.031049,0.256071,-0.205674,0.583755,0.268718,-0.608681,0.076159,-0.310388,0.024677,-0.323495,0.064168,0.600574,0.460265,0.089117,0.057191,-0.989311,0.057027,0.062312,0.147883,0.227977,-0.125187,-0.059058,0.167801,-0.835978,0.653512,-0.016134,0.659579,-0.12438,0.14926,-0.336267,-0.350397,0.04592,-0.485958,0.092552,-0.115349,-0.158546,0.325495,-0.215361,-0.042924,-0.048293,0.177941,-0.003278,-0.000834,0.006071,0.295335,0.11277,-0.181693,0.286427,-0.118223,-0.074186,-0.055841,0.045753,-1.82632,-0.080415,-0.048663],[-0.498405,-0.011659,-0.146136,0.362188,-0.80396,-0.181533,0.19261,-0.766514,-0.146849,0.341654,-0.113598,0.026179,-0.727308,-0.015566,0.111464,0.187965,0.337322,-0.047006,0.082928,0.044148,-0.326611,-0.210121,-0.062294,-0.058535,0.003113,-0.185991,-0.09685,0.361185,-0.540611,-0.209002,-0.02375,0.037805,0.069516,0.001482,-0.242313,0.21653,0.02625,-0.001584,0.077072,0.093437,-0.06565,-0.119345,-0.089669,-0.266162,0.137991,0.055091,-0.288476,0.515589,-0.133689,-0.55936,0.089573,0.099194,-0.038773,0.16407,-0.007992,-0.323905,-0.093199,0.091845,0.090468,0.452332,0.390812,0.356591,-0.059552,0.323458],[-0.53626,0.133169,0.209148,0.313362,0.300736,-0.061206,0.054307,0.146696,0.345845,0.265535,0.11067,0.436393,-0.567843,-0.103093,-0.918243,-0.071256,-0.066909,-0.181213,0.023072,-0.374289,0.159532,-0.189776,-0.498279,-0.013854,-0.092096,0.443755,0.398719,0.087251,0.115869,-0.63784,0.22287,-0.000239,-0.136581,0.191839,0.145602,-0.056527,0.573234,0.194375,0.34178,-0.200944,-0.185272,-0.253013,-0.541462,-0.556554,-0.469747,-0.646347,-0.192829,0.20148,0.217644,0.428669,-0.114179,-0.049052,0.038963,0.244371,0.295724,-0.004762,0.150471,0.061398,0.383729,-0.425977,0.31126,-0.067837,0.076298,-0.0955],[-0.118652,-0.505063,-0.33145,0.416648,0.277482,-0.230199,-0.5,0.332915,0.146252,0.486896,0.119958,-0.117956,-0.716814,0.11308,0.215728,-0.142125,-0.218657,-0.312106,0.251135,0.132027,0.355628,0.661352,0.11996,-0.058072,-0.265599,0.096222,-0.054772,-0.785658,0.037235,-0.159478,-0.378973,0.200771,-0.681324,-0.5699,-0.968483,-0.037398,-0.053084,-0.075853,-0.376646,0.142809,0.148442,0.122895,-0.008155,-0.158908,-0.459244,0.208537,0.388746,-1.244471,-2.131692,0.12357,0.04766,-0.066455,0.348945,0.092503,0.113759,0.401349,0.523614,-0.159097,-0.121376,0.161763,0.203527,0.170559,0.156923,0.098291],[0.316892,-0.30662,-0.676245,-0.16329,0.696891,-0.309031,-0.653354,-0.317626,0.341349,-0.211153,0.311864,-0.135862,0.159729,-0.074332,-0.209306,-0.19502,-0.052477,0.378581,0.602259,0.185817,-0.437562,-0.239476,0.429512,0.013983,-0.162843,0.308652,-0.312653,-0.271392,0.019426,0.437568,-0.697295,-0.588535,-0.432372,0.104471,0.223263,-0.088375,-1.607933,-0.723431,-0.330554,0.191289,-0.374046,-0.117012,0.502877,0.427329,-0.261922,0.620541,0.826024,-0.337586,-0.451768,0.006028,-1.144832,-0.153577,0.098691,-0.264471,-0.13446,0.219568,0.595991,0.195002,-0.707232,0.738101,-0.156301,0.163582,0.004704,0.124991],[0.22865,-0.02368,0.006577,0.210112,-0.080774,0.489433,0.013327,-0.050213,-0.010551,-2.849573,-0.102038,-0.435999,-0.17895,-0.016067,-0.246211,-0.130097,-0.531543,-0.051097,0.098913,0.143779,0.165407,0.207609,0.270463,-0.651394,-0.364697,-0.039453,-0.146216,-0.374394,0.240765,-0.146269,-0.024865,0.284444,0.107579,0.043445,-0.045727,-1.743537,0.016141,0.120588,-0.214668,-0.402621,0.085111,0.487902,-0.010011,-0.151497,0.344044,0.304009,0.257866,0.111822,-0.373878,0.557866,-0.303462,-0.028304,0.190383,-0.450432,0.260507,-0.292937,0.385888,-0.241438,0.02288,-0.284074,-0.819652,0.151519,-0.112998,-1.642955],[-0.257604,-0.040675,0.058585,-0.465146,0.246878,-1.410432,-0.035116,-0.230152,-0.398338,-0.467023,-0.583481,-0.500588,-0.026322,0.254382,-0.022759,0.538476,-0.471866,0.439461,-0.231625,-0.032356,0.131423,0.061577,-0.097372,0.043739,-0.054652,0.221577,0.015375,-0.180017,-0.740511,0.588744,-0.00272,-0.808326,-0.079589,0.23221,-0.13027,0.375046,-0.207619,0.323783,-0.344359,-0.012583,0.187185,0.009337,0.65625,0.509822,0.310502,0.194383,0.477498,0.47423,-0.231793,0.385252,-0.890685,0.157992,-0.006887,-0.322785,-0.398734,-0.033473,0.24552,0.447327,-0.268965,-0.211534,-0.066001,0.302298,-0.205055,-0.758442],[-0.22385,-0.09325,0.155077,-0.008194,-0.032149,0.451146,-0.211847,0.040314,0.557521,-0.420573,-0.314787,-0.192747,-0.051665,-0.223964,0.145222,-0.065431,-0.376626,-0.175721,0.218981,-0.237786,0.014947,0.03406,0.048429,0.263794,0.198415,0.658555,-0.119824,0.42358,0.240444,-0.418562,-0.361423,0.233684,-0.335641,-0.025005,-0.087516,0.246441,-0.245266,0.083535,-0.352764,0.505898,0.068528,0.601045,-0.430387,-0.309255,-0.550811,-0.751392,-0.135436,-0.35283,-1.83561,-0.461094,0.372501,0.152968,0.084376,-0.07238,0.233595,-0.263565,0.08553,-0.292445,-0.123383,0.153624,-2.15032,-0.280723,-0.26308,0.297965],[0.028549,-0.26968,0.028289,-0.120049,0.765208,-0.45077,-0.093389,0.107854,0.209747,-0.296148,-0.444272,-0.214958,0.080893,0.251985,0.202882,-0.069459,0.285748,-0.269502,0.003628,0.233193,-0.112392,-0.250271,0.007329,0.191769,0.23882,0.188452,0.067067,-0.568916,0.496877,0.00527,-0.440704,0.165139,0.562024,0.356829,-0.775699,0.201799,-0.336227,0.218126,-0.610265,-0.28944,0.054504,-0.015522,-0.858215,0.04429,0.462177,0.48516,0.144044,-0.448064,-2.212272,-0.009828,-0.21145,-0.032286,-0.237111,-0.284824,0.099907,-0.457114,0.282256,-0.007327,-0.405065,0.186214,0.21729,-0.072262,-0.025264,0.039432],[0.012097,-0.041034,-0.408953,0.519974,-0.438537,0.026752,0.249015,-0.065843,0.036177,0.492089,0.157694,-0.212093,0.120893,0.015887,-0.21531,0.344401,-0.182288,-0.219463,0.315773,-0.071067,-0.434696,-0.193654,0.762574,0.557262,0.674638,-1.137232,-0.065828,0.00771,-0.040329,0.111801,0.095643,-0.10481,0.497802,-0.213571,-0.241536,0.063575,-0.025225,0.032265,0.038512,-1.063042,-0.339864,0.712332,0.19695,0.264926,0.387056,-0.234073,0.190681,0.333757,0.403372,-0.304807,-3.701571,-0.876623,0.085798,0.382688,-0.136355,0.339857,0.603701,-0.192684,0.103674,-0.351051,-0.578876,0.078565,0.432834,-0.443428],[-0.402449,0.35684,0.13918,0.350036,0.053675,0.133727,-0.18741,0.642888,-0.16481,-0.263871,-0.248655,-0.852679,0.805384,-0.033314,-0.347764,-0.259008,-0.020731,-0.446449,0.531713,-0.034718,-0.501395,0.148013,0.571685,0.518385,0.315552,-0.15965,0.00678,0.042381,0.203976,-0.239846,0.313212,0.277756,-0.289951,-0.304035,0.098068,-0.254398,0.154216,0.003876,0.157311,0.137831,0.216299,0.025472,-0.816522,-0.037395,0.214663,0.125535,0.206313,0.437445,0.306036,-0.074529,0.357193,0.199083,-0.909486,-0.402299,-0.390861,-0.252988,-0.244806,-0.343227,0.383751,0.27634,0.44312,-0.39437,0.064917,-0.329985],[0.372486,0.09916,0.223227,-0.387839,-0.673349,0.061815,0.055124,-0.125134,0.417546,-0.246353,0.1488,0.192059,0.28865,0.066524,-0.028529,-0.274837,0.441779,0.15617,0.006602,-0.044424,0.946992,0.281522,0.087898,-0.627128,-0.151592,0.335318,0.02897,-0.376943,-1.645121,0.237051,0.013084,0.028985,-0.867403,0.451221,0.137281,0.120634,-0.03836,0.232487,0.038155,0.173524,0.25102,-0.353726,0.442049,0.216131,-0.350321,0.060539,-0.48372,-0.260017,0.191063,-0.611354,0.318818,0.024321,0.000251,-0.067961,0.077562,-0.217473,0.090125,0.06948,-0.031012,-0.119553,0.13599,0.496732,-1.115149,-0.235484],[0.481692,0.047292,0.146582,-0.106972,0.105684,0.209402,0.020783,-0.937221,0.082737,-0.337253,-0.138257,-0.176048,0.292997,0.552669,-0.151906,-0.182937,0.185742,0.062778,-0.080885,0.067382,0.313589,0.030949,0.34308,-0.022403,-0.578313,-0.51983,-0.112835,-0.472277,0.102899,0.087254,-0.023845,-0.306651,0.03013,-0.224791,-0.107089,-0.052939,-0.240666,-0.023521,-0.086161,0.079049,0.312309,-0.903663,-0.086673,0.094774,0.025717,0.129937,0.076168,-0.129677,0.11541,0.376495,0.172587,0.025335,0.136969,-0.147423,0.006219,0.263983,-0.368251,-0.144504,-0.153587,-0.405276,-0.849861,0.326177,-0.394923,0.01065],[0.741039,-0.458004,0.014005,0.212642,-1.536488,0.122112,-0.002553,1.124239,0.090721,-0.038877,-0.026716,-0.761877,0.664413,-0.105635,-0.423846,-0.048621,0.211123,-1.036913,-0.599674,0.032406,0.56456,0.042176,-0.445076,-0.919889,0.207082,0.002763,0.038898,-0.349402,-0.872996,0.148952,-0.31088,-0.017825,-0.335134,-0.727295,-0.178645,0.167907,-0.148958,0.117369,0.075579,0.374994,-0.131292,-0.31208,-0.249658,0.049126,-0.895651,0.100002,0.194195,-0.333213,0.77138,-0.664151,0.488057,-0.911095,0.201489,-0.108828,-0.170184,-0.971498,-0.667632,-0.711608,-0.283289,-0.147148,0.049866,-0.878984,-0.531806,-0.213555],[0.635635,-0.024986,0.105622,-0.172664,-0.948982,-0.081685,-0.018673,0.220983,-0.674337,-0.22224,0.251375,0.197805,-0.209652,-0.262372,-0.057662,0.028232,0.316693,-0.171854,0.55493,-0.111206,-0.477106,0.079999,-0.424099,0.252859,-0.282503,-0.072421,-0.085053,-0.034874,0.39522,-0.173731,-0.089147,0.06158,0.344016,-0.012989,0.162604,-0.173448,0.172945,-0.252157,0.147071,-0.201251,-0.089216,-0.399098,-0.215444,-0.139927,-0.087401,-0.005428,-0.53083,-0.163272,0.238011,0.015559,-0.348898,0.052633,-0.162437,0.287526,-0.680851,0.336625,0.112756,-0.001153,0.148028,-0.164462,-0.144706,-0.158895,-0.138552,-0.731522],[0.349257,-0.276511,-0.084906,0.557429,0.15968,0.409418,0.101341,-0.184944,0.055499,0.417389,0.076496,-0.385023,0.209378,-0.088598,0.143298,-0.50107,-0.33104,-0.026335,-0.078011,0.087425,-0.300852,-0.40095,-0.123089,-0.749164,0.23467,-0.278103,-0.008983,0.25249,0.436607,-0.016059,-0.281725,-0.135446,0.247147,0.12027,-0.080631,-1.0701,-0.036519,-0.110887,-0.005159,0.20886,-0.36412,0.18066,0.644912,0.013319,-0.230955,0.478466,0.14939,0.085912,-0.319447,0.002335,0.018696,-0.080058,0.22859,0.136161,-0.487169,0.367379,0.579953,-0.220446,-0.154074,0.55988,-1.453193,0.107279,0.323979,0.389558],[-0.059141,-0.677537,-0.460257,-0.039903,0.033013,-0.024328,0.053779,-0.671078,0.488389,0.539348,0.427701,0.116088,0.499782,0.066438,-1.127329,0.437588,-0.309168,0.301569,0.332959,-0.064173,0.325716,0.419235,0.098167,-0.303013,0.159442,-0.2255,-0.093688,0.312183,-0.87455,-0.057873,-0.512841,-0.49674,0.019201,0.073506,-0.688005,0.530319,-0.09685,0.159761,-0.370237,-0.502629,-0.127369,0.116278,-0.167085,-0.053839,-0.062247,0.660298,0.058161,0.063122,-0.03794,0.023727,0.642032,-0.5833,0.146254,-0.283863,0.068481,-0.226017,-0.281074,-0.080104,-0.196917,0.41116,0.120683,0.076219,-0.164704,0.22821],[-0.075232,-0.160636,0.027663,-0.226791,0.196578,0.190798,-0.067619,0.109639,0.35734,0.339746,-0.186415,-0.186253,-0.069361,0.234065,0.243227,-0.500853,-0.549334,0.140879,-0.23152,0.046369,-0.161973,-0.102208,-0.002064,0.085191,-0.703008,-0.324488,0.038052,-0.069592,0.234513,0.171774,-0.204104,-0.39234,-0.083823,-0.302308,0.125956,0.044654,-0.132167,0.177365,0.038462,0.023845,0.209395,0.244233,0.070813,0.137961,0.02152,-0.034098,0.292757,-0.494138,-0.005265,0.342819,-0.084902,0.280655,0.062256,0.566464,-0.100995,-0.386678,-0.160653,0.189797,-0.120237,0.329411,-1.812626,0.027711,-0.122016,0.21276],[0.185144,-1.200033,-0.578884,0.418982,-0.403397,-0.213674,-0.041141,0.922911,-0.525958,0.662056,-0.426926,0.024352,1.141383,0.061871,1.608271,-0.285761,-0.172794,-0.292278,-0.599131,0.109935,-1.50429,0.607336,-0.46855,-1.900154,0.189289,-0.862006,-1.094223,0.97951,-0.176556,0.151028,-1.082655,-0.760418,0.32076,-0.119596,-0.496778,0.213766,-2.15016,-0.752045,-0.500678,-0.027951,0.323703,0.882249,0.566429,0.003426,-0.295623,-0.045025,0.742082,0.284922,-0.239842,0.620682,-0.094755,0.427924,0.093658,-0.191743,-1.171852,-0.139364,0.573454,-0.566296,-1.890322,-0.324445,1.129304,-0.085633,-0.221687,0.394833],[-0.403378,-0.895056,-0.461669,-0.263182,0.008374,0.075894,0.129876,0.018895,-0.141984,-0.324835,0.121368,0.008315,-0.002361,0.238654,0.519525,0.202103,0.075654,-0.551354,-0.394674,-0.141498,0.006111,-0.52863,-0.045872,0.045584,0.231115,0.14185,0.490923,0.096174,-0.332574,-0.957862,-1.088461,0.15798,0.183504,0.255849,-0.897238,-0.534518,0.262871,-0.633946,0.531605,-0.432071,-0.005158,-0.494816,0.360458,-0.955143,0.12045,-0.448157,-0.019136,0.178262,-1.179495,0.23044,-0.344015,-0.035323,0.071105,0.114865,-0.21939,0.41075,-0.264727,0.324574,-0.233875,0.155556,0.243777,0.22655,-0.199994,-0.267356],[-0.968583,0.50019,0.261594,0.1546,-0.457217,0.409399,-0.215254,-0.119899,0.16772,0.111327,-0.08922,-0.740097,0.436991,-0.174328,-0.364102,0.082364,0.274012,-1.040895,-0.022552,-0.888582,0.536487,0.725756,0.233066,0.168624,-1.252698,0.355011,0.074811,0.032205,0.15865,-1.869937,0.339232,0.276755,0.119655,0.037674,0.593728,0.627942,0.22784,0.327952,-0.131544,0.116201,0.097787,0.20559,-0.347347,-1.399169,-0.802445,0.63364,0.218201,0.195669,0.08259,-0.13379,0.405797,-0.345842,0.110445,-0.262425,0.239243,0.063736,-1.282978,-0.20068,0.391397,0.45643,0.061415,-0.489836,0.346252,0.03802],[-0.466517,0.390636,0.346299,0.580049,0.594984,-0.580289,0.080622,-0.029045,-0.107486,-0.130587,-0.298866,0.012415,-0.604355,-0.405817,-1.379142,0.269986,0.14227,0.048486,0.160398,0.065019,0.164461,0.094658,-1.197618,-0.868171,0.999527,0.133378,0.01017,0.045293,0.220847,-0.002892,0.206054,0.37783,-0.164064,0.385019,0.014678,-0.195792,0.170919,-0.020833,0.087921,-0.327084,0.062706,-0.591046,-0.037764,-0.033371,-0.365025,-0.585345,0.066093,0.370706,0.555905,0.024329,-0.608216,-0.173742,-0.0226,0.283651,0.286881,0.068211,-0.145613,-0.210125,0.232074,-1.207406,-0.150358,0.133056,0.070502,-0.579853],[0.237606,-0.234337,-0.210678,-0.158165,-1.016485,0.173432,-0.259372,0.73523,0.029826,-0.209262,-0.272346,0.082043,-0.059783,0.035219,0.265154,-0.00262,-0.2543,-0.377957,-0.072954,-0.004387,-0.264923,-0.353616,-0.204068,-0.369284,-0.101181,0.216322,-0.007518,-0.256239,-0.146762,-0.236332,-0.077656,0.150716,-0.506192,0.138256,-0.388856,0.199812,-0.421339,-0.023934,-0.13852,0.49958,0.243095,-0.617392,-0.21538,-0.194053,0.120233,-0.125768,0.183987,-0.122109,0.210085,0.41627,0.333809,0.112928,0.19981,-0.258416,0.21874,-0.391623,0.302989,0.268539,-0.344732,-0.134294,0.542854,-0.107195,-0.143693,-0.187652],[0.310014,0.010193,-0.159915,0.119055,-0.115292,-0.80208,0.133408,0.144135,0.402906,0.087773,0.568926,0.095843,-0.170702,-0.30539,-0.588935,-0.063304,0.164571,0.113759,-0.028238,-0.008851,-0.112385,0.13607,0.117488,0.444709,-0.209636,0.218728,-0.147433,-0.140626,-0.138197,0.129479,0.068123,-0.351889,0.137389,0.229957,-0.087245,-0.569968,0.133508,0.115365,0.241578,-0.142426,-0.260717,-0.009146,-0.73515,0.091058,-0.393598,0.122315,-0.680242,0.144826,-0.193848,-0.987924,0.135231,-0.117505,-0.068862,-1.053952,-0.00708,0.063653,0.323781,-0.082438,0.225154,0.215698,-0.006643,0.009914,-0.077747,-0.048985],[0.436176,-0.249841,-0.048544,-0.23239,0.359791,-0.447191,-0.207799,-0.509009,0.452552,0.100632,-0.129897,-0.149381,0.045666,0.458642,-0.104428,0.42938,-0.179467,0.362094,0.039233,-0.105933,0.522492,-0.233479,-0.640588,0.4799,-0.321073,-0.01611,-0.130946,-0.385458,0.224809,0.386584,-0.406964,-0.019054,0.424082,-0.521768,0.039689,0.09351,-0.99546,0.160255,-0.682282,0.354993,0.111905,0.15186,-0.296366,0.322664,0.143125,0.331925,0.471897,0.149125,-0.458765,0.580607,-1.073587,0.237289,-0.169509,0.428653,0.202792,0.392807,0.31482,0.234962,-0.72136,0.478186,0.346669,-0.344946,-0.879669,0.33047],[-0.022938,0.281862,0.087683,-0.082473,-0.179433,-0.322139,0.138069,0.315059,-0.299466,-0.275512,0.38211,0.460942,-0.001945,0.125573,0.594133,0.693905,-0.25161,-0.24574,0.209581,-0.059252,-0.017764,0.397365,-0.394154,-0.478603,-0.092151,0.364361,0.05896,0.291319,-1.22864,-0.309905,0.290049,0.591493,0.29678,-0.307331,-0.277815,0.197239,0.087888,-0.066536,0.294526,-0.433736,0.050042,0.011466,0.535358,-0.229696,0.337768,-0.372732,-0.154131,0.306951,0.52728,0.012508,-0.537299,0.28776,0.084642,0.226835,0.164395,0.22134,0.135777,-0.098691,0.113792,-0.076417,-0.191978,0.339012,0.2063,0.082268],[0.073209,0.137048,-0.28781,0.361214,-0.369874,0.008173,0.341933,-0.128875,0.146436,-0.435041,-0.068906,-0.016442,0.235203,0.154367,0.163569,-0.241586,-0.055624,0.107462,0.118561,-0.034774,-0.119009,0.440692,0.15798,-0.280601,0.007127,0.113033,-0.008707,0.170021,0.226357,0.212816,0.30347,0.771678,0.048362,-0.068782,-0.149916,0.037671,0.490414,-0.258938,0.582261,-0.777562,0.105253,0.08607,0.40039,0.169785,0.243022,-0.547906,0.32698,0.071068,0.172554,-0.610421,-1.384839,0.197388,0.047102,0.060216,-0.007912,0.07254,-0.263085,0.450091,0.432984,-0.59277,-0.912573,-0.350976,-0.143582,0.018438],[-0.636886,0.092396,-0.05902,-0.167815,-0.561698,-0.0748,-0.073934,0.298187,-0.065968,0.329707,0.375709,0.187082,-0.177028,0.487025,0.005982,-0.425287,-0.525173,0.031334,0.23893,-0.07047,-0.43128,-0.129994,-2.012219,-0.036286,0.391103,-0.179527,0.017202,-0.050593,-0.107972,0.012534,0.101516,0.217776,-1.269617,0.455933,0.220247,0.00785,0.067176,0.016462,-0.481541,-0.026349,0.30696,0.040315,0.219732,0.017072,-0.379188,0.539223,-0.0995,-0.050598,0.126027,0.196655,0.109596,-0.040533,-0.143385,-0.018109,0.379285,0.210822,-0.814698,-0.149404,0.120544,0.480373,-0.316038,-0.084606,0.601816,-0.068102],[0.08223,-0.015686,0.069468,-0.615199,0.029318,-0.986943,-0.014712,0.391484,-0.09346,0.125413,-0.099267,-0.361632,0.083165,0.049105,0.25959,-0.050729,0.017668,-0.047003,-0.182762,-0.119288,0.43249,-0.266975,-1.758564,0.695335,-0.207897,0.174034,-0.067493,-0.064323,-2.286659,-0.094185,-0.049817,-0.014549,-0.11419,0.555235,-0.10874,0.003816,-0.177643,0.125089,0.269336,-0.316234,0.214392,0.070525,-0.222212,-0.029666,0.098834,0.33198,-0.286667,-0.427126,0.084354,-0.461805,0.023197,0.026164,-0.122778,0.140329,-0.277997,-0.105015,0.098102,-0.054304,-0.097333,0.342692,0.224366,0.061713,0.256394,-0.074589],[0.396112,0.079416,-0.031761,-0.328512,0.180608,0.052607,-0.059506,-1.146336,0.701681,-0.042768,-0.411846,-0.343514,0.405875,0.641897,0.304025,-0.286399,-0.339493,-0.155239,0.172447,0.119254,0.083066,-0.783182,0.979141,-0.264333,-0.010858,-0.101199,0.007994,-0.115826,0.498039,0.029028,0.092573,0.395976,0.472771,0.47945,0.137847,0.418904,0.076217,-0.281241,-0.301956,0.513472,-0.141635,-0.031123,0.250816,0.070174,-0.084448,0.003565,0.35619,-0.721266,-0.028561,-0.511326,0.01767,-0.130185,-0.407959,-0.081921,0.367371,-0.113118,0.249801,-0.10494,0.055076,-0.154409,-0.693483,0.411959,-0.087836,0.514937],[0.063768,0.043107,0.043745,-0.103519,-0.181373,-0.582323,0.111055,-0.035329,-0.899178,0.145622,-0.30474,-0.982118,-0.441833,0.21676,-0.036019,0.167055,0.379063,0.10182,0.131049,0.021814,-0.39982,-0.222777,0.123911,-0.116595,0.109739,0.031322,0.161299,-0.277793,0.265217,-0.106375,0.010478,0.446596,0.304143,-0.150106,0.243114,0.086532,0.134212,0.235656,0.55052,-0.148747,0.080325,0.069323,-0.736134,-0.109153,-0.267604,0.372848,0.157567,-0.34098,-0.304255,0.175406,0.070299,-0.618619,0.043783,0.02583,-0.09698,0.222838,0.081296,-0.178398,0.019056,0.351297,-0.478713,-0.405593,-0.287125,-0.016288],[-0.170021,0.063361,0.090331,0.021791,-0.064715,-1.052772,-0.067711,0.05599,0.038373,-0.254446,-0.651683,0.026147,-0.142936,-0.010996,0.193769,0.556001,0.250191,0.042233,0.063172,-0.095408,0.306104,-0.069131,-2.461782,0.29631,-0.611686,-0.952575,0.100033,-0.313068,0.246021,0.145234,-0.014691,-0.22916,-0.459095,0.133044,0.350264,0.126287,-0.03452,0.012987,-0.442261,-0.331707,0.073704,-0.230143,-0.332359,0.213601,0.012435,-0.668111,-0.371675,-0.383828,0.29388,-0.321553,0.557283,-0.052836,-0.450639,-0.219638,-0.366254,-0.364272,0.026898,-0.254685,0.0053,0.243919,-0.128827,-0.207732,0.110636,0.053486],[0.220837,0.484182,0.351762,0.100306,-1.092567,0.553602,0.165646,0.211644,-0.317091,0.16463,0.187693,-0.41975,-0.054512,-0.233521,-0.384893,-0.44745,0.365174,-0.334488,-0.465374,0.043104,0.050254,-0.764716,0.126195,0.031452,0.258965,-0.000213,-0.018096,-0.169296,-0.327917,-0.456672,0.410109,0.251412,-0.008206,0.034531,0.322802,0.186835,0.257654,-0.233601,0.217903,-0.103662,0.073588,0.226982,-0.352346,-0.419191,0.109479,-0.431834,0.029255,0.346889,0.028486,-0.769924,0.284433,-0.182951,-0.216349,-0.113412,0.204221,0.342271,-0.288631,-0.038089,0.206964,-0.129672,0.330889,-0.113497,0.292723,0.456316],[-0.491476,-0.267592,-0.108915,-0.076675,-0.027683,-1.123249,-0.154344,0.151443,-0.016851,0.207653,0.258134,0.122033,0.10681,-0.078712,-0.167157,-0.17632,-0.055432,0.140089,-0.275233,0.090748,0.008128,-0.204479,-0.476747,-0.566224,-1.389547,-0.070838,-0.090818,0.14181,0.206042,0.59642,-0.11577,-0.618321,-0.279954,-0.159473,-0.039308,0.08862,-0.016095,0.046018,-0.23487,-0.979141,0.098621,-0.950021,0.178774,0.63556,-0.257913,0.135237,0.332067,0.038076,0.429683,-0.340021,-1.341873,-0.099462,-0.036767,-0.264304,-0.257606,0.014914,0.129249,0.306568,-0.043095,-0.467324,-0.669215,-0.115571,-0.438658,0.379047],[0.086371,0.110163,-0.192631,-0.573998,-0.040826,-0.667069,0.15286,0.282131,-0.326759,-0.278496,0.075075,0.667477,-0.186333,0.57813,-0.182798,0.332044,0.036217,-0.093803,0.241437,-0.141008,0.254363,-0.086028,0.017648,-0.459808,0.358306,0.167172,0.100686,0.07081,0.29346,-0.708009,0.162636,0.542196,-0.014342,-0.183936,0.189291,0.209151,0.637676,-0.259949,-0.124029,-0.156174,0.525855,0.3491,-0.225817,-0.691746,0.075498,-0.432039,-0.28319,-0.585086,-0.015299,0.064281,0.10133,-0.378392,-0.157929,0.220724,0.146398,0.461069,-0.117191,-0.156352,0.519095,-0.100581,-1.350075,0.291949,0.164497,-0.10675]],"b":[-0.204078,-1.006759,0.237111,-0.102505,-0.025402,0.126078,0.250013,0.176353,-0.198415,-0.127577,0.449735,0.340867,-0.743709,0.230304,-0.046253,0.145234,0.123808,0.254807,-0.17368,0.728793,-0.199557,-0.612816,-0.18897,0.017346,0.062198,0.69774,0.813089,0.283331,-0.421867,-0.640503,0.155247,-0.104461,-0.142229,0.007733,-0.320799,-0.51429,-0.398376,-0.635381,0.648604,-1.185942,0.253016,-0.013378,0.153129,-0.216912,-0.065052,-0.771691,-0.405748,0.236904,-0.307209,0.312213,-0.948426,-0.585708,-0.383768,0.442217,0.002281,0.458274,-0.384789,0.535321,0.523231,-0.347793,0.214807,0.504213,0.062815,-0.271637]},{"W":[[-0.626697,-0.007888],[0.380861,-0.493028],[-0.035258,0.218118],[0.413162,0.001395],[-1.489011,0.000885],[0.711681,0.005769],[-0.002888,0.096802],[-0.607648,-0.014588],[-0.181462,-1e-05],[-0.369225,-0.001106],[-0.278772,-0.020331],[-0.288127,-0.009169],[-0.432797,0.019619],[0.081475,-0.004133],[-0.559366,-0.029014],[-0.14534,0.003534],[-0.280655,-3.6e-05],[0.088365,-0.1898],[0.135357,0.037074],[-0.014017,-0.158136],[0.287387,0.002862],[0.644295,0.005068],[-0.713173,0.008857],[0.560365,0.004316],[-0.35657,-0.000922],[-0.249371,-0.01218],[0.013069,0.14284],[0.265336,0.006157],[1.045154,-0.002207],[-0.166886,0.736688],[-0.187153,0.325421],[0.32234,-0.005402],[0.501698,0.00463],[0.27667,0.0043],[0.537796,0.038831],[-0.339801,-0.002205],[-0.078689,-0.264424],[-0.102199,-0.068032],[0.417493,0.008144],[-1.106573,-0.0281],[0.077361,0.021846],[0.344856,0.002319],[-0.386895,-0.007005],[-0.132187,-0.660212],[-0.245939,-2.1e-05],[-0.797018,0.005951],[-0.243833,0.004948],[0.294008,0.010361],[-0.293481,-0.041145],[-0.304764,-0.008646],[-0.632575,-0.012196],[-0.047852,0.081381],[-0.003053,0.019301],[0.198114,0.003375],[0.233351,0.012964],[-0.271223,0.001359],[-0.336913,0.024167],[-0.031528,-0.075009],[0.004395,0.278567],[-0.249242,-0.001142],[1.387952,0.004045],[-0.181492,0.004245],[0.422383,0.004614],[-0.75401,-0.000919]],"b":[0.364591,-0.050918]}]}'

In [ ]:
# VYHODNOTENIE A MODELY-PORADCOVIA (src/walkforward.py)
"""Vyhodnotenie poradcu na histórii: pre každý počiatočný rok XY model odpovie len z dát známych do XY.

Pre každý XY a horizont H:
  1. skutočný víťaz  = ktorý scenár mal po H rokoch väčší majetok (model.tier2_grid, skutočné dáta XY+1..XY+H),
  2. odpoveď modelu  = „kúpa“ / „prenájom“ vypočítaná z dát <= XY,
  3. porovnanie      = podiel rokov, v ktorých mal model pravdu (spolu so základnými stratégiami
                        „vždy kúpa“, „vždy prenájom“).

Modely:
  odhad2 – ten istý, ale polovica odhadu sa vracia k dlhodobému priemeru (MEAN_REVERSION).
  odhad  – „sústava malých modelov“ (poznámka 1B): z posledných L rokov histórie odhadne rast cien,
           rast nájmov a výnos akcií, nasimuluje oba scenáre s týmito konštantnými hodnotami a porovná.
  sieť   – neurónová sieť z webu, dostane tie isté odhadnuté vstupy (orezané na rozsah, v ktorom sa učila).
Nič z budúcnosti XY nevstupuje do odpovede (okrem parametrov siete – viď správa, kapitola o obmedzeniach).
"""
import json
import numpy as np
import pandas as pd


LOOKBACK = 20                         # dĺžka okna histórie pre odhad (určená vopred, nelaďovaná na výsledkoch)
HORIZONS = (10, 20, 30)
MEAN_REVERSION = 0.5                  # druhý model: polovica odhadu sa vracia k dlhodobému priemeru (zvolené až po prvom behu, viď správa)
RATE_SPREAD = 0.015                   # marža hypotéky nad dlhodobou sadzbou (rovnako ako v tier2_grid)


def known_inputs(d: pd.DataFrame, year: int, lookback: int = LOOKBACK, shrink: float = 0.0) -> dict:
    """Odhady z dát <= year. d: ročná tabuľka s indexom year (stĺpce housing_capgain, housing_rent_yd, eq_tr, ltrate).

    shrink = 0: čistý priemer z posledných `lookback` rokov. shrink = w > 0: podiel w sa „vráti k dlhodobému
    priemeru“ (všetky roky v dátach až do `year`, od roku 1901) – ochrana proti extrapolácii boomu."""
    full = d.loc[:year]
    h = full.tail(lookback + 1)
    geo = lambda x: float(np.exp(np.log1p(x).mean()) - 1)

    def est(df):
        g = df["housing_capgain"].iloc[1:]
        r = (df["housing_rent_yd"] / df["housing_rent_yd"].shift(1) * (1 + df["housing_capgain"]) - 1).iloc[1:]
        return geo(g), max(geo(r), 0.0), geo(df["eq_tr"].iloc[1:])
    g, rent_g, eq = est(h)
    if shrink:
        g2, r2, e2 = est(full.dropna(subset=["housing_capgain", "housing_rent_yd", "eq_tr"]))
        g, rent_g, eq = [(1 - shrink) * a + shrink * b for a, b in ((g, g2), (rent_g, r2), (eq, e2))]
    return {
        "house_growth": g, "rent_growth": rent_g, "equity_return": eq,
        "gross_yield": float(h["housing_rent_yd"].iloc[-1]),
        "mortgage_rate": max(float(h["ltrate"].iloc[-1]) / 100 + RATE_SPREAD, 0.01),
    }


def advise_estimate(inp: dict, years: int, p=None) -> float:
    """Model „odhad“: výhoda kupujúceho po `years` rokoch pri konštantných odhadnutých hodnotách (> 0 = kúpa)."""
    from dataclasses import replace
    p = replace(p or SPAIN, gross_yield=inp["gross_yield"])
    n = years
    adv = simulate(np.full(n, inp["house_growth"]), np.full(n, inp["rent_growth"]),
                   np.full(n, inp["equity_return"]), np.full(n, inp["mortgage_rate"]), p, 1, n)
    return float(adv[-1])


# ------------------------------------------------------------------ neurónová sieť (numpy)
RANGES = {"gross_yield": (0.01, 0.07), "house_growth": (-0.05, 0.20), "rent_growth": (0.0, 0.20),
          "equity_return": (0.0, 0.20), "mortgage_rate": (0.01, 0.20)}   # záloha; skutočné rozsahy sú v súbore s váhami


def load_net(path_or_dict):
    w = path_or_dict if isinstance(path_or_dict, dict) else json.load(open(path_or_dict))
    return {k: (np.array(v) if k in ("mu", "sd") else v) for k, v in w.items()}


def advise_net(net: dict, inp: dict, years: int, p=None):
    """Vráti (P(kúpa vyhráva), počet vstupov, ktoré bolo treba orezať na tréningový rozsah)."""
    p = p or SPAIN
    clipped = 0
    v = {}
    for k, (lo, hi) in net.get("ranges", RANGES).items():
        v[k] = min(max(inp[k], lo), hi)
        clipped += v[k] != inp[k]
    x = np.array([years, p.ltv, v["gross_yield"], p.buy_cost, v["house_growth"], v["rent_growth"],
                  v["equity_return"], v["mortgage_rate"]], float)
    a = (x - net["mu"]) / net["sd"]
    for i, L in enumerate(net["layers"]):
        a = a @ np.array(L["W"]) + np.array(L["b"])
        if i < len(net["layers"]) - 1:
            a = np.maximum(a, 0)
    return float(1 / (1 + np.exp(-a[0]))), int(clipped)


# ------------------------------------------------------------------ vyhodnotenie
def evaluate(annual: pd.DataFrame, net: dict | None = None, horizons=HORIZONS, first=1950, lookback=LOOKBACK):
    """annual: tabuľka ako spain_annual_1870_2025.csv. Vracia DataFrame: jeden riadok na (rok, horizont)."""
    actual = tier2_grid(annual, start_from=first, max_years=max(horizons))
    d = annual.set_index("year").sort_index()
    rows = []
    for H in horizons:
        for y in actual.index:
            if H not in actual.columns or pd.isna(actual.loc[y, H]):
                continue
            inp = known_inputs(d, int(y), lookback)
            adv_est = advise_estimate(inp, H)
            adv_mr = advise_estimate(known_inputs(d, int(y), lookback, MEAN_REVERSION), H)
            row = {"start": int(y), "H": H, "actual_adv": float(actual.loc[y, H]), "buy_won": bool(actual.loc[y, H] > 0),
                   "est_adv": adv_est, "est_buy": adv_est > 0, "mr_adv": adv_mr, "mr_buy": adv_mr > 0, **{"in_" + k: v for k, v in inp.items()}}
            if net is not None:
                pb, clipped = advise_net(net, inp, H)
                row.update(net_p=pb, net_buy=pb > 0.5, net_clipped=clipped)
            rows.append(row)
    return pd.DataFrame(rows)


def summarize(ev: pd.DataFrame) -> dict:
    out = {}
    for H, g in ev.groupby("H"):
        s = {"n_years": int(len(g)), "first": int(g.start.min()), "last": int(g.start.max()),
             "buy_won_share": float(g.buy_won.mean()),
             "always_buy": float(g.buy_won.mean()), "always_rent": float(1 - g.buy_won.mean()),
             "estimate": float((g.est_buy == g.buy_won).mean()),
             "mean_reversion": float((g.mr_buy == g.buy_won).mean())}
        if "net_buy" in g:
            s["net"] = float((g.net_buy == g.buy_won).mean())
            ind = g[g.net_clipped == 0]
            s["net_in_domain"] = {"n": int(len(ind)), "acc": float((ind.net_buy == ind.buy_won).mean()) if len(ind) else None}
        for a, b, nm in ((1950, 1979, "1950-79"), (1980, 1999, "1980-99"), (2000, 2020, "2000+")):
            k = g[(g.start >= a) & (g.start <= b)]
            if len(k):
                s.setdefault("by_period", {})[nm] = {"n": int(len(k)), "buy_won": float(k.buy_won.mean()),
                                                     "estimate": float((k.est_buy == k.buy_won).mean()),
                                                     "mean_reversion": float((k.mr_buy == k.buy_won).mean())}
        out[int(H)] = s
    return out


# ------------------------------------------------------------------ majetok v oboch scenároch pre zvolený rok
def scenario_wealth(annual: pd.DataFrame, start: int, horizon: int, price: float = 250_000, p=None) -> dict:
    """Majetok oboch domácností po `horizon` rokoch pre nákup v roku `start` (skutočné dáta), v eurách.

    nominal = v eurách roku start+horizon, real = v eurách roku `start` (po odpočítaní inflácie)."""
    buyer, renter, adv, defl = tier2_run(annual, start, horizon, parts=True, p=p)
    f = lambda x: float(x[-1] * price)
    return {"start": start, "horizon": horizon, "price": price,
            "nominal": {"hypoteka": f(buyer), "prenajom": f(renter), "rozdiel": f(adv)},
            "real": {"hypoteka": f(buyer / defl), "prenajom": f(renter / defl), "rozdiel": f(adv / defl)},
            "winner": "hypotéka" if adv[-1] > 0 else "prenájom"}


def advice_today(annual: pd.DataFrame, net: dict | None, years: int, year: int | None = None, overrides: dict | None = None) -> dict:
    """Rada pre dnešok: odhady z posledných LOOKBACK rokov (alebo vlastné `overrides`) -> obe odpovede."""
    d = annual.set_index("year").sort_index()
    inp = known_inputs(d, int(year or d.index.max()))
    inp.update(overrides or {})
    adv = advise_estimate(inp, years)
    out = {"inputs": inp, "years": years, "estimate_adv": adv, "estimate_answer": "hypotéka" if adv > 0 else "prenájom"}
    if net is not None:
        pb, clipped = advise_net(net, inp, years)
        out.update(net_p_buy=pb, net_answer="hypotéka" if pb > 0.5 else "prenájom", net_clipped_inputs=clipped)
    return out


net = load_net(json.loads(NN_JSON))

## A · Majetok v oboch scenároch pre zvolený počiatočný rok
Zmeň `START` (1950–2015), `HORIZON` (roky) a `PRICE` (cena bytu v €).

In [ ]:
START, HORIZON, PRICE = 1990, 10, 250_000

w = scenario_wealth(annual, START, HORIZON, PRICE)
print(f"Nákup v roku {START}, byt za {PRICE:,.0f} €, horizont {HORIZON} rokov")
print(f"{'':12}{'v eurách roku '+str(START+HORIZON):>22}{'v eurách roku '+str(START):>22}")
for k, nm in (('prenajom', 'Podnájom'), ('hypoteka', 'Hypotéka'), ('rozdiel', 'Rozdiel (H−P)')):
    print(f"{nm:12}{w['nominal'][k]:>20,.0f} €{w['real'][k]:>20,.0f} €")
print('Víťaz:', w['winner'].upper())

## B · Rada modelu pre zadanú situáciu
`YEARS` = koľko rokov plánuješ bývať. Model si odhady (rast cien, nájmov, akcií, sadzbu) vezme z posledných 20 rokov dát; môžeš ich prepísať v `OVERRIDES` (napr. vlastný očakávaný rast cien).

In [ ]:
YEARS = 15
OVERRIDES = {}   # napr. {'house_growth': 0.04, 'gross_yield': 0.05, 'mortgage_rate': 0.03}

a = advice_today(annual, net, YEARS, overrides=OVERRIDES)
print('Vstupy modelu:', {k: round(v, 4) for k, v in a['inputs'].items()})
print(f"Model „odhad z histórie“: {a['estimate_answer'].upper()}  (výhoda hypotéky {a['estimate_adv']:+.2f} ceny bytu)")
print(f"Neurónová sieť:           {a['net_answer'].upper()}  (P(hypotéka vyhrá) = {a['net_p_buy']:.0%}; orezaných vstupov: {a['net_clipped_inputs']})")

## C · Vyhodnotenie na histórii
Pre každý počiatočný rok XY (v ktorom celé obdobie leží v dátach) sa spočíta, ktorý scenár skutočne vyhral, a porovná s odpoveďou modelu, ktorá využíva **len dáta do roku XY**. Tabuľka ukazuje podiel rokov, v ktorých mal model pravdu, spolu s jednoduchými stratégiami „vždy hypotéka“ a „vždy podnájom“.

In [ ]:
ev = evaluate(annual, net)
sm = summarize(ev)
print(f"{'horizont':>9}{'roky':>6}{'vždy hyp.':>11}{'vždy prenáj.':>14}{'odhad':>8}{'návrat k pr.':>14}{'sieť':>7}")
for H, s in sm.items():
    print(f"{H:>7} r.{s['n_years']:>6}{s['always_buy']:>11.0%}{s['always_rent']:>14.0%}{s['estimate']:>8.0%}{s['mean_reversion']:>14.0%}{s['net']:>7.0%}")
print()
print("Sieť dostane tie isté odhadnuté vstupy ako model „odhad z histórie“ (rozsahy vstupov pokrývajú aj vysokú infláciu).")

In [ ]:
# graf: skutočná výhoda hypotéky pre horizont 10 rokov a odpoveď modelu
import matplotlib.pyplot as plt
g = ev[ev.H == 10]
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.bar(g.start, g.actual_adv, color=['tab:orange' if b else 'tab:blue' for b in g.buy_won])
ok = (g.est_buy == g.buy_won)
ax.scatter(g.start[ok], [0]*ok.sum(), marker='o', s=14, color='green', label='odhad správne', zorder=3)
ax.scatter(g.start[~ok], [0]*(~ok).sum(), marker='x', s=22, color='red', label='odhad nesprávne', zorder=3)
ax.set_title('Výhoda hypotéky po 10 rokoch (oranžová = vyhrala hypotéka, modrá = podnájom)')
ax.set_xlabel('počiatočný rok'); ax.set_ylabel('podiel ceny (reálne)'); ax.legend(); plt.show()